# ACDC experiment: reconstruction, verification and dynamic shimming

Run this notebook **from top to bottom**. All experiment parameters are in the next cells;
no configuration JSON is read. Each stage displays its input, result and checks.

Python reconstructs the Twix data, reads the NPZ calibration and draws diagnostics.
**The forward model and optimization execute the existing MATLAB functions.**
The final cells export one waveform CSV after the numerical and waveform-data checks pass.

The example input folder is `acquisition_001`: a mono-echo 2D GRE/FLASH reference Twix,
a multi-echo B0 Twix, a 32-channel ACDC volume and an EPI scan. A conventional
mono-echo GRE scan is also supported. Each acquisition has its own Pulseq `.seq` file.
The reference, B0 and scan may use different matrix sizes and FOVs when their physical
grids cover the selected ROI. The simulation object is refined to at least the scan
pixel spacing in step 4a before resampling B0 and ACDC, preventing coarse-grid point-source artifacts. The ROI is defined from the reconstructed reference
in step 4b and displayed before checking map coverage.

Figures stay in the notebook. The final exports create numbered CSVs (`waveform1.csv`, `waveform2.csv`, ...) in the output folder; no FPGA control is performed.


In [ ]:
# 1. Imports and paths
%matplotlib inline
import hashlib
import os
import subprocess
import tempfile
from datetime import datetime
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat, savemat
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "Experiment").is_dir() else Path.cwd().parent
EXPERIMENT_DIR = PROJECT_ROOT / "Experiment"
sys.path.insert(0, str(EXPERIMENT_DIR))

# Refresh local readers when rerunning this cell in an existing kernel.
import importlib
import read_acdc_sequence
import prepare_acdc_inputs
importlib.reload(read_acdc_sequence)
importlib.reload(prepare_acdc_inputs)

from prepare_acdc_inputs import read_gre, read_scan, check_reference_sequence, check_physical_geometry
from read_acdc_sequence import read_sequence
from read_acdc_b0 import read_b0_coil_images, b0_from_coil_images
from read_acdc_calibration import read_calibration
from resample_acdc_maps import resample_to_grid
import prepare_acdc_simulation_grid
importlib.reload(prepare_acdc_simulation_grid)
from prepare_acdc_simulation_grid import prepare_simulation_reference
from run_acdc_experiment import check_calibration, current_windows
from export_acdc_waveform import export_waveform, quantize_currents

INPUT_DIR = PROJECT_ROOT / "acquisition_014"
FILES = {
    "reference": next(INPUT_DIR.rglob("*reference.dat")),
    "b0": next(INPUT_DIR.rglob("*b0_map.dat")),
    "scan": next(INPUT_DIR.rglob("*scan.dat")),
    "calibration": INPUT_DIR / "ACDC_3T.npz",
    "scan_sequence": INPUT_DIR / "sequences/gre_epi_64x64_fov200mm_nav3_rise1ms_ext1_single_trigger_first_readout_v4.seq",
    "reference_sequence": INPUT_DIR / "sequences/gre_32x32_fov200mm_tr1s.seq",
    "b0_sequence": INPUT_DIR / "sequences/b0_map.seq",
}
SESSION = datetime.now().strftime("notebook_%Y%m%d_%H%M%S")
OUTPUT_DIR = EXPERIMENT_DIR / "acdc_output" / SESSION
# Python/MATLAB working files and worker storage share this scratch folder.
SCRATCH_DIR = Path("/scratch/mv1289")
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
_workspace = tempfile.TemporaryDirectory(prefix="acdc_notebook_", dir=SCRATCH_DIR)
WORK_DIR = Path(_workspace.name)
print("Input folder:", INPUT_DIR)
print("CSV output folder (waveform1.csv, waveform2.csv, ...):", OUTPUT_DIR)
print("Temporary MATLAB workspace:", WORK_DIR)

## 2. Experiment parameters

These values are explicit choices for this dataset. `b0_sign` is applied **once** after
phase-based field estimation. The receive-coil phase convention and the absolute frequency
branch need independent experimental verification. Connected B0 components may have
unknown relative multiples of `1 / echo_spacing`.

The Bay5 registration and source hash below retain your previous confirmation.
The numerical current bounds are the values you entered. The final cell only places currents in
CSV columns and timestamps; it never connects to the controller.
`CURRENT_RESPONSE_ENABLED` selects the first-order ACDC response for EPI.
`CURRENT_RESPONSE_TAU_US` is either one time constant shared by all 32 channels or
32 values in calibration order. The default is `133 / ln(100)` microseconds.
MATLAB integrates the real current from the RF center to every ADC sample, preserving
all previous transitions. The measured 0-to-1 A step supplies an approximation to
validate experimentally; a common tau assumes identical channel dynamics.
Set the flag to `False` to recover the previous instantaneous model. After any change,
rerun from the parameter cell, including optimization; an old solution cannot be reused.

For elapsed time `dt` after a DAC change, the model is:

$$I_c(dt)=I_{new,c}+(I_{old,c}-I_{new,c})\exp(-dt/\tau_c).$$

`I_old` is the real current reached at the change. The EPI phase is then:

$$\phi_{ACDC}(x,y,t)=-2\pi\sum_c b_c(x,y)\int_0^t I_c(s)\,ds,$$

where `b_c` is the measured calibration in Hz/A and time is in seconds from the RF center.


In [ ]:
# Reconstruction and physical registration
REFERENCE_THRESHOLD = 0.1
NAVIGATOR_PHASE_BRANCH = 0  # 0: small odd/even mismatch; 1: alternative pi branch.
SHOW_MEAN_B0_DIAGNOSTIC = True  # Display only; does not change the ACDC comparison.
B0_SIGN = 1
GEOMETRY_CONFIRMED = True
B0_SIGN_CONFIRMED = True
T2STAR_S = np.inf  # No measured T2* map: retain the existing no-decay assumption.
B0_PARAMETERS = {
    "magnitude_fraction": 0.05,
    "max_phase_residual_rad": 0.5,
    "min_component_pixels": 10,
    "frequency_offset_hz": 0.0,
}
CALIBRATION_PARAMETERS = {
    "units": "Hz/A",
    "registration_confirmed": True,
    "patient_to_source_m": np.eye(4).tolist(),
    "source_sha256": "bace6a584e780d80af8f2ca7c12957bd5b9dc7462be0968f949cc9a99243a06c",
    "registration_note": "Bay5 coils and alignment confirmed by the user on 2026-10-03.",
}

# MATLAB simulation and optimization
MATLAB_EXECUTABLE = Path("/usr/pubsw/packages/matlab/26.1/bin/matlab")  # Replace on another computer.
RUN_FORWARD_MODEL = True
RUN_OPTIMIZATION = True
IMAGE_INITIALIZATION = "static"  # "static": use the static shim result; "zero": use zero DAC commands.
FIELD_LIMIT_HZ = 500.0
MAX_CURRENT_A = 3.0
TOTAL_CURRENT_LIMIT_A = 30.0
MAX_ITERATIONS = 150
WORKERS = 1
THREADS_PER_WORKER = 1
DEVICE = "gpu"

# ACDC temporal response (EPI only). Tau is in microseconds here.
CURRENT_RESPONSE_ENABLED = True
CURRENT_RESPONSE_TAU_US = 133.0 / np.log(100.0)  # 99% of a 0-to-1 A step in 133 us.
# To use measured channel values, replace the scalar above with 32 positive values
# in calibration channel order, for example: [28.9, 30.0, ...].
# Assumption: actual currents are zero before the first imaging command.
assert isinstance(CURRENT_RESPONSE_ENABLED, (bool, np.bool_)), "Use True or False."
tau_us = np.asarray(CURRENT_RESPONSE_TAU_US, dtype=float)
if tau_us.ndim == 0:
    tau_us = np.full(32, float(tau_us))
assert tau_us.shape == (32,) and np.isfinite(tau_us).all() and (tau_us > 0).all(),     "CURRENT_RESPONSE_TAU_US needs one positive scalar or 32 positive channel values."
CURRENT_RESPONSE = {"enabled": CURRENT_RESPONSE_ENABLED, "tau_s": tau_us * 1e-6}

# CSV column order and numerical current limits.
WAVEFORM_PARAMETERS = {
    "channel_map": [i for i in range(1, 33)],  # Enter 32 distinct physical output numbers in 1..64, in calibration order.
    "channel_polarity": [1] * 32,
    "max_current_A": MAX_CURRENT_A,
    "total_current_limit_A": TOTAL_CURRENT_LIMIT_A,
    "min_dac_interval_s": None,  # Optional positive minimum spacing between CSV rows.
}
RF_DELAY_S = 0.0  # 0: times from the first RF center; enter the measured delay for trigger-relative times.
EXPORT_WAVEFORM = True

In [ ]:
# Display diagnostics in the notebook without writing additional output files.
def show_figure(fig):
    plt.show()
    plt.close(fig)


def show_table(rows, headings=("Parameter", "Value")):
    lines = ["| " + " | ".join(headings) + " |", "| " + " | ".join(["---"] * len(headings)) + " |"]
    lines += ["| " + " | ".join(str(value) for value in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def image_panel(ax, values, title, cmap="gray", **kwargs):
    artist = ax.imshow(values, cmap=cmap, interpolation="nearest", **kwargs)
    ax.set_title(title)
    ax.set_xlabel("Array column")
    ax.set_ylabel("Array row")
    return artist


for name, path in FILES.items():
    assert path.is_file(), f"Missing {name}: {path}"
assert not isinstance(B0_SIGN, bool) and B0_SIGN in (-1, 1), "B0_SIGN must be +1 or -1."
assert 0 <= REFERENCE_THRESHOLD < 1, "REFERENCE_THRESHOLD must be in [0, 1)."
assert np.isscalar(T2STAR_S) and T2STAR_S > 0, "T2STAR_S must be positive; infinity is allowed."
show_table([(name, path.name) for name, path in FILES.items()])

## 3. Sequence trajectory and acquisition timing

The trajectory is derived from the actual RF/ADC/gradient events. The `Matrix`, `FOV`,
`TE` and `TR` definitions are checked against these events. Cartesian order is used
for reconstruction, while `line_index` preserves chronological current order.
For spoiled GRE every line starts a new transverse phase history; EPI retains one RF history.

In [ ]:
# Read the scan and reference sequences independently.
acquisition, timing = read_sequence(FILES["scan_sequence"])
assert not CURRENT_RESPONSE_ENABLED or timing["sequence_kind"] == "epi", "Disable the temporal response for a GRE scan; this model currently supports EPI."
reference_acquisition, reference_timing = read_sequence(FILES["reference_sequence"])
show_table([
    ("Scan kind", timing["sequence_kind"]),
    ("Scan matrix [Nx, Ny]", timing["matrix_size"]),
    ("Scan TE / TR [ms]", np.array([acquisition["TE_s"], acquisition["TR_s"]]) * 1000),
    ("ADC dwell [us]", acquisition["dwell_s"] * 1e6),
    ("Reference matrix [Nx, Ny]", reference_timing["matrix_size"]),
    ("Reference TE / TR [ms]", np.array([reference_acquisition["TE_s"], reference_acquisition["TR_s"]]) * 1000),
])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
plot = axes[0].scatter(acquisition["kx_mInv"], acquisition["ky_mInv"],
                       c=acquisition["time_s"] * 1000, s=6, cmap="viridis")
axes[0].set(xlabel="kx [cycles/m]", ylabel="ky [cycles/m]", title="ADC trajectory, colored by time")
fig.colorbar(plot, ax=axes[0], label="Time from first RF center [ms]")
plot = image_panel(axes[1], acquisition["line_index"], "Chronological line index", cmap="viridis")
fig.colorbar(plot, ax=axes[1])
starts, ends = current_windows(timing)
for line, (start, end) in enumerate(zip(starts, ends)):
    axes[2].plot([start * 1000, end * 1000], [line + 1, line + 1], color="tab:blue")
for center in np.unique(acquisition["rf_time_s"]):
    axes[2].axvline(center * 1000, color="tab:red", alpha=0.3)
axes[2].set(xlabel="Time from first RF [ms]", ylabel="Current state", title="Current windows and RF centers")
show_figure(fig)

## 4. Reconstruct the reference and verify its physical coordinates

The reference is a **mono-echo Cartesian 2D GRE/FLASH Twix** with one RF excitation
per acquired line. Its own `reference_gre.seq` supplies the expected matrix, FOV,
TE, TR, ADC dwell, readout polarity and acquisition order. These are checked
against the raw data before defining the ROI; no image size is entered manually.

Reorder the acquired k-space lines, apply a centered inverse Fourier transform to
each receive coil, then combine magnitudes with root-sum-of-squares (RSS).
The array axes are `[phase row, readout column]`. Physical coordinates use
`X = readout` and `Y = phase`, in metres. The displayed pixel coordinates also
allow you to inspect the reference geometry before resampling the measured maps.

The RSS magnitude is an effective transverse-amplitude proxy, not a proton-density
or T1 map. The next cell defines the ROI from this reconstructed reference.


In [ ]:
# Reconstruct the raw reference and validate it against its own GRE sequence.
gre, reference_diagnostics = read_gre(
    FILES["reference"], reference_acquisition, return_intermediates=True)
check_reference_sequence(gre, reference_acquisition)
reference_magnitude = np.asarray(gre["magnitude"], float)
reference_metadata = gre["twix_metadata"]
X_m, Y_m = np.asarray(gre["X_m"]), np.asarray(gre["Y_m"])
show_table([
    ("Reference matrix [phase rows, readout columns]", reference_magnitude.shape),
    ("Receive coils", reference_metadata["receiver_count"]),
    ("TE / TR [ms]", np.array([reference_metadata["TE_s"], reference_metadata["TR_s"]]) * 1000),
    ("ADC dwell [us]", reference_metadata["dwell_s"] * 1e6),
    ("Readout / phase FOV [mm]", np.asarray(reference_metadata["FOV_m"]) * 1000),
    ("Slice center [patient mm]", reference_metadata["slice_position_mm"]),
])
fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
image_panel(axes[0], np.log1p(np.linalg.norm(reference_diagnostics["kspace"], axis=1)), "Reference: log k-space RSS")
image_panel(axes[1], np.abs(reference_diagnostics["coil_images"][:, 0, :]), "Reference: receiver 1 magnitude")
image_panel(axes[2], reference_magnitude, "Reference: combined RSS magnitude")
show_figure(fig)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
plot = image_panel(axes[0], np.angle(reference_diagnostics["coil_images"][:, 0, :]),
                   "Reference: receiver 1 phase [rad]", cmap="twilight", vmin=-np.pi, vmax=np.pi)
fig.colorbar(plot, ax=axes[0])
axes[1].scatter(X_m * 1000, Y_m * 1000, c=reference_magnitude, s=8, cmap="gray")
axes[1].set(xlabel="X: readout [mm]", ylabel="Y: phase [mm]", title="Physical pixel coordinates", aspect="equal")
show_figure(fig)


### 4a. Match the object sampling to the scan

The MATLAB forward model sums point sources at the object pixels. A 32×32
object sampled directly into 64×64 k-space produces a dotted comb, even at
zero B0, and biases the image optimization. Before selecting the ROI, refine
the reference grid to at least the scan's physical pixel spacing on both axes.
For equal 200 mm FOVs, a 32×32 reference and 64×64 EPI use a 64×64 simulation grid.
Different FOVs are handled by physical spacing, rather than matrix size alone.

Linear interpolation retains the reference's physical FOV and Fourier origin.
It does not add measured spatial resolution. Samples outside reference coverage
are excluded. B0 and all 32 ACDC responses are then interpolated directly from
their measured source grids onto this simulation grid, with their validity masks.
Patient position and pixel spacing are updated together for ACDC placement.
The native GRE reconstruction remains available as `native_reference_magnitude`
and `gre`; the following ROI, fits and comparisons use the interpolated reference.
After changing an acquisition, rerun from the top, including all optimizations.


In [ ]:
# Keep the native measured reference; use a sufficiently fine object for MATLAB.
native_reference_magnitude = np.asarray(gre["magnitude"], float).copy()
simulation_reference, simulation_grid_report = prepare_simulation_reference(gre, acquisition)
reference_magnitude = simulation_reference["magnitude"]
X_m, Y_m = simulation_reference["X_m"], simulation_reference["Y_m"]
reference_grid_valid = simulation_reference["valid_mask"]
show_table([
    ("Native measured GRE [phase rows, readout columns]", native_reference_magnitude.shape),
    ("Scan [phase rows, readout columns]", np.asarray(acquisition["kx_mInv"]).shape),
    ("Simulation object [rows, columns]", reference_magnitude.shape),
    ("Native / simulation spacing [row, column mm]", f'{simulation_grid_report["native_spacing_mm"]} / {simulation_grid_report["simulation_spacing_mm"]}'),
    ("Reference interpolation needed", simulation_grid_report["refined"]),
    ("Simulation pixels covered by the measured reference", f"{int(reference_grid_valid.sum())}/{reference_grid_valid.size}"),
])
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
reference_display_scale = native_reference_magnitude.max()
image_panel(axes[0], native_reference_magnitude, "Native measured GRE", vmin=0, vmax=reference_display_scale)
image_panel(axes[1], reference_magnitude, "Reference on simulation grid", vmin=0, vmax=reference_display_scale)
show_figure(fig)


### 4b. Define and inspect the ROI

The default selects all reference pixels above `REFERENCE_THRESHOLD * maximum_magnitude`.
The threshold is set in step 2. Optional X/Y limits select a rectangle in physical
millimetres; keep both limits at `None` to select the whole thresholded object.

Change these limits and rerun this cell and the following cells. Green pixels in the
overlay are the requested ROI. The binary mask and excluded object pixels show exactly
what was selected. The computed mask stays in memory and is included in the internal MATLAB input.
No mask is read from the input folder. B0 coverage is checked separately in step 8;
unreliable field measurements never shrink the requested ROI automatically.

In [ ]:
# Optional physical rectangle limits; None means no restriction on that axis.
ROI_X_LIMITS_MM = None  # Example: (-40.0, 40.0), X = readout.
ROI_Y_LIMITS_MM = None  # Example: (-40.0, 40.0), Y = phase.

threshold_support = reference_grid_valid & (reference_magnitude > REFERENCE_THRESHOLD * reference_magnitude.max())
support = threshold_support.copy()
for axis_name, limits, coordinates in (
    ("X", ROI_X_LIMITS_MM, X_m), ("Y", ROI_Y_LIMITS_MM, Y_m)
):
    if limits is not None:
        bounds = np.asarray(limits, dtype=float)
        assert bounds.shape == (2,) and np.isfinite(bounds).all() and bounds[0] <= bounds[1], f"{axis_name} limits need two finite increasing values in mm."
        support &= (coordinates * 1000 >= bounds[0]) & (coordinates * 1000 <= bounds[1])

support_before_validity_filter = support.copy()
excluded_object_pixels = threshold_support & ~support
excluded_signal_fraction = reference_magnitude[~support].sum() / reference_magnitude.sum()
fig, axes = plt.subplots(2, 2, figsize=(11, 9), layout="constrained")
image_panel(axes[0, 0], reference_magnitude, "GRE reference magnitude")
image_panel(axes[0, 1], support, f"Computed ROI: {int(support.sum())} pixels", vmin=0, vmax=1)
image_panel(axes[1, 0], reference_magnitude, "GRE overlay: green = selected ROI")
axes[1, 0].imshow(np.ma.masked_where(~support, support), cmap="Greens", vmin=0, vmax=1, alpha=0.45, interpolation="nearest")
if support.any() and not support.all():
    axes[1, 0].contour(support, levels=[0.5], colors=["lime"], linewidths=0.8)
image_panel(axes[1, 1], excluded_object_pixels, "Thresholded object pixels outside ROI", cmap="Reds", vmin=0, vmax=1)
show_figure(fig)
show_table([
    ("Reference magnitude threshold", REFERENCE_THRESHOLD),
    ("X / Y limits [mm]", f"{ROI_X_LIMITS_MM} / {ROI_Y_LIMITS_MM}"),
    ("Thresholded object pixels", int(threshold_support.sum())),
    ("Selected ROI pixels", int(support.sum())),
    ("Reference magnitude excluded", f"{excluded_signal_fraction:.1%}"),
])
assert support.any(), "The computed ROI is empty: inspect the threshold and physical limits above."
ROI_PARAMETERS = {
    "method": "reference magnitude threshold with optional physical X/Y limits",
    "reference_threshold": REFERENCE_THRESHOLD,
    "x_limits_mm": [] if ROI_X_LIMITS_MM is None else list(ROI_X_LIMITS_MM),
    "y_limits_mm": [] if ROI_Y_LIMITS_MM is None else list(ROI_Y_LIMITS_MM),
}
print("ROI defined in memory; inspect the selection figure before continuing.")

## 5. Scan k-space, navigator correction and RSS reconstruction

The raw scan is reconstructed independently of the reference. Negative readouts
are reversed using `REFLECT`; a centered inverse FFT and RSS combine the receivers.
Before/after magnitude images share one intensity scale.

For three equally spaced `+,-,+` navigators, estimate the phase differences
`navigator 2 - navigator 1` and `navigator 2 - navigator 3` separately. Their
average removes approximately linear B0 phase evolution between the echoes.
Do **not** average the complex positive echoes: they can cancel almost completely.
Each receiver's affine phase fit uses only its strongest contiguous region with
reliable signal in all three navigators; noisy background is not phase-unwrapped.

The correction has a **pi branch ambiguity**. `NAVIGATOR_PHASE_BRANCH = 0` assumes
a small odd/even phase mismatch at the strongest signal sample; `1` selects the
alternative branch. Verify this convention experimentally. Neither the GRE nor
image registration is used to choose the branch. Constant B0 during the three
navigators is assumed. The sequence check rejects asymmetric navigator spacing.

Inspect the selected fit pixels, phase residuals and cancelled-average diagnostic
below. Receivers with fewer than four contiguous reliable pixels keep zero
correction and are reported explicitly. This step preserves the physical EPI B0
displacement and distortion; it does not shift the image to match the reference.

A good navigator phase fit does not guarantee an artifact-free image. A uniform
B0 offset also displaces the EPI along the phase axis and can wrap the object
across the image boundary. Section **8b** checks this separately using the
measured field map. A weaker half-FOV ghost may remain after both operations.


In [ ]:
# Inspect reconstruction before and after navigator correction.
# Reload and rebind the reader without recreating the MATLAB workspace.
import importlib
import prepare_acdc_inputs
read_scan = importlib.reload(prepare_acdc_inputs).read_scan
print("Reconstruction reader:", Path(prepare_acdc_inputs.__file__).resolve())

scan_image, scan_metadata, scan_diagnostics = read_scan(
    FILES["scan"], acquisition, return_intermediates=True,
    navigator_phase_branch=NAVIGATOR_PHASE_BRANCH)
check_physical_geometry(gre, scan_metadata, acquisition, support, X_m, Y_m, dicom_input=False)
for key in ("TE_s", "TR_s", "dwell_s"):
    tolerance = 1e-9 if key == "dwell_s" else 1e-6
    assert abs(scan_metadata[key] - acquisition[key]) <= tolerance, f"Scan header disagrees on {key}."
assert scan_metadata["navigator_count"] == timing["navigator_count"], "Navigator count mismatch."
assert np.array_equal(scan_metadata["polarity"], acquisition["polarity"]), "Readout polarity mismatch."
assert scan_metadata["acquired_phase_rows"] == np.argsort(acquisition["line_index"][:, 0]).tolist(), "Acquisition order mismatch."
nav_scale = max(scan_diagnostics["uncorrected_image"].max(), scan_image.max())
fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
image_panel(axes[0], np.log1p(np.linalg.norm(scan_diagnostics["kspace"], axis=1)), "Raw reordered k-space: log RSS")
image_panel(axes[1], np.log1p(np.linalg.norm(scan_diagnostics["corrected_kspace"], axis=1)), "Corrected k-space: log RSS")
image_panel(axes[2], scan_diagnostics["uncorrected_image"], "RSS before navigator correction", vmin=0, vmax=nav_scale)
image_panel(axes[3], scan_image, "RSS after navigator correction", vmin=0, vmax=nav_scale)
show_figure(fig)
if scan_metadata["navigator_count"]:
    nav = scan_diagnostics["navigator_hybrid"]
    nav_fit = scan_diagnostics["navigator_fit"]
    fitted = nav_fit["fitted_channels"]
    fit_pixels = nav_fit["fit_mask"].sum(axis=1)
    skipped_receivers = (np.flatnonzero(~fitted) + 1).tolist()
    show_table([
        ("Navigator phase branch", NAVIGATOR_PHASE_BRANCH),
        ("Receivers with a valid phase fit", f"{int(fitted.sum())}/{len(fitted)}"),
        ("Receivers left uncorrected", skipped_receivers),
    ])
    if fitted.any():
        retention = nav_fit["outer_average_retention"][nav_fit["fit_mask"]]
        show_table([
            ("Selected fit pixels per receiver [min, max]", [int(fit_pixels[fitted].min()), int(fit_pixels[fitted].max())]),
            ("Median phase fit residual [rad]", float(np.median(nav_fit["fit_residual_rad"][fitted]))),
            ("Median amplitude retained by complex outer average", f"{np.median(retention):.1%}"),
        ])
        receiver = int(np.argmax(np.where(fitted, np.abs(nav).sum(axis=(0, 2)), -1)))
        columns = np.arange(scan_image.shape[1])
        selected = nav_fit["fit_mask"][receiver]
        fig, axes = plt.subplots(1, 3, figsize=(17, 4), layout="constrained")
        for index in range(3):
            axes[0].plot(columns, np.abs(nav[index, receiver]), label=f"Navigator {index + 1}")
        axes[0].plot(columns, np.abs((nav[0, receiver] + nav[2, receiver]) / 2),
                     "k--", label="Complex outer average (unused)")
        axes[0].axvspan(columns[selected][0], columns[selected][-1], color="green", alpha=0.12, label="Fit region")
        axes[0].set(xlabel="Readout column", ylabel="Magnitude", title=f"Navigator signal: receiver {receiver + 1}")
        axes[0].legend(fontsize=8)
        for pair, label in enumerate(("Phase 2 - 1", "Phase 2 - 3")):
            axes[1].plot(columns[selected], nav_fit["pair_phase_rad"][pair, receiver, selected], ".-", label=label)
        axes[1].plot(columns[selected], nav_fit["phase_estimate_rad"][receiver, selected], "k.", label="Mean phase, selected branch")
        axes[1].plot(columns[selected], scan_diagnostics["navigator_phase_correction_rad"][receiver, selected], "k-", label="Affine fit")
        axes[1].set(xlabel="Readout column", ylabel="Phase [rad]", title="Separate pair phases and fitted correction")
        axes[1].legend(fontsize=8)
        image_panel(axes[2], nav_fit["fit_mask"], "Reliable contiguous pixels used for each fit", vmin=0, vmax=1)
        axes[2].set(xlabel="Readout column", ylabel="Receiver index")
        show_figure(fig)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
    image_panel(axes[0], np.angle(nav[:, 0]), "Navigator phases, receiver 1", cmap="twilight", vmin=-np.pi, vmax=np.pi)
    plot = image_panel(axes[1], scan_diagnostics["navigator_phase_correction_rad"], "Fitted phase correction [rad]", cmap="RdBu_r")
    axes[1].set(xlabel="Readout column", ylabel="Receiver index")
    fig.colorbar(plot, ax=axes[1])
    show_figure(fig)
print("Scan matrix:", scan_image.shape, "; receiver count:", scan_metadata["receiver_count"])


## 6. Multi-echo B0: complex images before field estimation

B0 requires phase information from at least two equally spaced monopolar echoes.
First inspect each echo's k-space, RSS magnitude and one receiver's phase. This cell
runs before masking or phase unwrapping, so acquisition/reconstruction problems remain visible.

In [ ]:
# Read complex echoes without yet estimating a field.
b0_raw = read_b0_coil_images(FILES["b0"], FILES["b0_sequence"])
coils = b0_raw["coil_images"]  # [echo, receiver, phase row, readout column]
te_s = np.asarray(b0_raw["metadata"]["sequence_info"]["TE_s"])
assert abs(b0_raw["metadata"]["slice_position_mm"][2] - gre["position_mm"][2]) <= 1e-3, "B0 and reference slice planes differ."
assert np.isclose(abs(np.dot(b0_raw["metadata"]["slice_quaternion_wxyz"], scan_metadata["slice_quaternion_wxyz"])), 1, atol=1e-8), "B0 and scan orientations differ."
show_table([("B0 matrix", coils.shape[2:]), ("Echoes", coils.shape[0]),
            ("Receive coils", coils.shape[1]), ("Echo times [ms]", te_s * 1000)])
ne = coils.shape[0]
fig, axes = plt.subplots(3, ne, figsize=(5 * ne, 11), layout="constrained", squeeze=False)
for echo in range(ne):
    image_panel(axes[0, echo], np.log1p(np.linalg.norm(b0_raw["kspace"][echo], axis=0)), f"Echo {echo + 1}: log k-space RSS")
    image_panel(axes[1, echo], np.linalg.norm(coils[echo], axis=0), f"Echo {echo + 1}: magnitude RSS")
    image_panel(axes[2, echo], np.angle(coils[echo, 0]), f"Echo {echo + 1}: receiver 1 phase [rad]", cmap="twilight", vmin=-np.pi, vmax=np.pi)
show_figure(fig)

## 7. Phase differences, reliability masks and measured field

For each receiver, `echo[n+1] * conj(echo[n])` cancels its static receive phase. The
products are summed across receivers and echo pairs. The field convention is
`signal(TE) = amplitude * exp(-2π i B0 TE)`, followed by the explicit scanner sign choice.

Reliability requires sufficient magnitude, consistent adjacent-echo phase increments,
and components large enough for spatial phase unwrapping. No B0 value is extrapolated.
Each component is anchored at its strongest pixel in the principal frequency band;
this assumption does not measure the absolute frequency branch.

In [ ]:
# Show phase increments before spatial unwrapping.
echo_products = np.sum(coils[1:] * coils[:-1].conj(), axis=1)
wrapped_increment = np.angle(echo_products.sum(axis=0))
fig, axes = plt.subplots(1, ne, figsize=(5 * ne, 4), layout="constrained", squeeze=False)
for pair in range(ne - 1):
    image_panel(axes[0, pair], np.angle(echo_products[pair]), f"Echo pair {pair + 1}: wrapped increment", cmap="twilight", vmin=-np.pi, vmax=np.pi)
image_panel(axes[0, -1], wrapped_increment, "Combined wrapped increment [rad]", cmap="twilight", vmin=-np.pi, vmax=np.pi)
show_figure(fig)

In [ ]:
# Estimate the field with the same tested reconstruction as the pipeline.
b0_result = b0_from_coil_images(coils, te_s, return_intermediates=True, **B0_PARAMETERS)
b0_result["b0Hz"] *= B0_SIGN
b0_result["X_m"], b0_result["Y_m"] = b0_raw["X_m"], b0_raw["Y_m"]
b0_details = b0_result["intermediates"]
fig, axes = plt.subplots(2, 3, figsize=(15, 9), layout="constrained")
image_panel(axes[0, 0], b0_details["magnitude_mask"], "Magnitude threshold passed")
image_panel(axes[0, 1], b0_details["phase_residual_mask"], "Echo consistency passed")
image_panel(axes[0, 2], b0_result["mask"], "Final reliability mask")
plot = image_panel(axes[1, 0], b0_result["phase_residual_rad"], "Echo increment disagreement [rad]", cmap="magma", vmin=0, vmax=np.pi)
fig.colorbar(plot, ax=axes[1, 0])
plot = image_panel(axes[1, 1], b0_details["component_labels"], "Connected field components", cmap="tab20")
plot = image_panel(axes[1, 2], b0_result["b0Hz"], "Measured B0 [Hz], masked", cmap="RdBu_r")
fig.colorbar(plot, ax=axes[1, 2])
show_figure(fig)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
image_panel(axes[0], b0_details["quality_before_component_filter"], "Magnitude and phase checks passed")
image_panel(axes[1], b0_result["mask"], "Large reliable components retained")
image_panel(axes[2], b0_details["quality_before_component_filter"] & ~b0_result["mask"], "Small components removed")
show_figure(fig)
fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.hist(b0_result["b0Hz"][b0_result["mask"]], bins=60)
ax.set(xlabel="Measured B0 [Hz]", ylabel="Reliable pixels", title="Masked field distribution")
show_figure(fig)
show_table([("Reliable native-grid pixels", int(b0_result["mask"].sum())),
            ("Frequency band at component anchors [Hz]", b0_result["metadata"]["principal_frequency_band_hz"]),
            ("B0 sign applied", B0_SIGN)])

## 8. Resample measured maps onto the simulation grid

The measured B0 grid and simulation grid are independent. Linear interpolation uses their physical
coordinates and accepts a target pixel only when every contributing source measurement is
valid. The full reference support and the selected ROI are both compared against coverage.
The requested ROI is kept unchanged. Missing B0 measurements are displayed explicitly
and block simulation until this coverage problem is resolved.

In [ ]:
# Interpolate without filling holes or extending the measured field.
b0_on_reference, b0_valid = resample_to_grid(
    b0_result["b0Hz"], b0_result["X_m"], b0_result["Y_m"], b0_result["mask"], X_m, Y_m)
fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
image_panel(axes[0], threshold_support, "Full reference support")
image_panel(axes[1], b0_valid, "Valid B0 on reference grid")
image_panel(axes[2], support, "Selected simulation support")
plot = image_panel(axes[3], np.where(b0_valid, b0_on_reference, np.nan), "Measured B0 on reference [Hz]", cmap="RdBu_r")
fig.colorbar(plot, ax=axes[3])
show_figure(fig)
missing_b0_pixels = support & ~b0_valid
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
image_panel(ax, reference_magnitude, "Requested ROI: red = missing reliable B0")
ax.imshow(np.ma.masked_where(~missing_b0_pixels, missing_b0_pixels), cmap="Reds", vmin=0, vmax=1, alpha=0.65, interpolation="nearest")
if support.any() and not support.all():
    ax.contour(support, levels=[0.5], colors=["lime"], linewidths=0.8)
show_figure(fig)
show_table([("Full support covered by B0", f"{int((threshold_support & b0_valid).sum())}/{int(threshold_support.sum())}"),
            ("Selected ROI covered by B0", f"{int((support & b0_valid).sum())}/{int(support.sum())}"),
            ("Selected ROI pixels missing reliable B0", int(missing_b0_pixels.sum())),
            ("Reference signal excluded by ROI", f"{excluded_signal_fraction:.1%}")])
assert GEOMETRY_CONFIRMED and B0_SIGN_CONFIRMED, "Confirm the physical geometry and scanner sign in the parameter cell."
#assert not missing_b0_pixels.any(), f"Reliable B0 is missing for {int(missing_b0_pixels.sum())}/{int(support.sum())} requested ROI pixels. The ROI has not been reduced; inspect the coverage figures before simulation."

## 8b. Diagnostic: remove the measured mean B0 offset

This optional display separates **frequency displacement and wrapping** from
the remaining odd/even ghost. Average only the reliable B0 pixels inside the
selected ROI. For EPI, a uniform offset predicts a phase-axis displacement of
approximately `mean_B0_Hz * echo_spacing_s * number_of_phase_rows` pixels.

Our signal convention is `exp(-2j*pi*B0_Hz*time_s)`. Multiply the **raw** imaging
samples and the three raw navigators by its inverse at their actual ADC times,
then refit the navigator phase. Negative-readout sample times must be reversed
in the same way as the Twix samples. The sequence already provides reordered
imaging times; navigator times are recovered from their centers and dwell.

The figures use one intensity scale. This removes one uniform frequency only;
spatial B0 distortion, signal loss and a residual half-FOV ghost can remain.
It assumes that the measured B0 sign, frequency branch and geometry are valid.
No GRE image registration or image-based fitting is used.

**This is an offline diagnostic, not ACDC shimming.** Its output is named
`mean_b0_diagnostic_image`; `scan_image`, the MATLAB inputs, the CSV exports and
the measured hardware comparison/RMSE keep their original reconstruction.
The unshimmed field map is not applied to `scan_corr.dat`, whose field changes
when the ACDC currents start.


In [ ]:
# Reconstruct a separate diagnostic image; preserve scan_image for hardware evaluation.
mean_b0_diagnostic_image = None
if SHOW_MEAN_B0_DIAGNOSTIC and timing["sequence_kind"] == "epi":
    import prepare_acdc_inputs
    centered_fft = prepare_acdc_inputs.centered_fft
    estimate_epi_navigator_phase = prepare_acdc_inputs.estimate_epi_navigator_phase

    measured_roi = support & b0_valid & np.isfinite(b0_on_reference)
    assert measured_roi.any(), "No reliable B0 pixels in the selected ROI."
    mean_b0_hz = float(np.mean(b0_on_reference[measured_roi]))
    std_b0_hz = float(np.std(b0_on_reference[measured_roi]))
    ny, nx = scan_image.shape
    predicted_shift_pixels = mean_b0_hz * acquisition["echo_spacing_s"] * ny

    # time_s follows the same phase/readout order as the raw k-space.
    sample_times = np.asarray(acquisition["time_s"])
    assert sample_times.shape == (ny, nx), "ADC times must match the scan matrix."
    diagnostic_kspace = scan_diagnostics["kspace"] * np.exp(
        2j * np.pi * mean_b0_hz * sample_times)[:, None, :]
    diagnostic_hybrid = centered_fft(diagnostic_kspace, (2,), inverse=True)

    # Apply the same frequency change to navigators BEFORE estimating their phase.
    if scan_metadata["navigator_count"]:
        sample_offsets = (np.arange(nx) - (nx - 1) / 2) * acquisition["dwell_s"]
        navigator_times = np.asarray(acquisition["navigator_echo_center_s"])[:, None] + np.stack(
            (sample_offsets, -sample_offsets, sample_offsets))  # +,-,+ after REFLECT reordering.
        diagnostic_nav_kspace = scan_diagnostics["navigator_kspace"] * np.exp(
            2j * np.pi * mean_b0_hz * navigator_times)[:, None, :]
        diagnostic_nav = centered_fft(diagnostic_nav_kspace, (2,), inverse=True)
        diagnostic_phase, diagnostic_nav_fit = estimate_epi_navigator_phase(
            diagnostic_nav, phase_branch=NAVIGATOR_PHASE_BRANCH)
        diagnostic_hybrid[scan_diagnostics["negative_readout_rows"]] *= np.exp(
            -1j * diagnostic_phase)[None, :, :]

    diagnostic_coils = centered_fft(diagnostic_hybrid, (0,), inverse=True)
    mean_b0_diagnostic_image = np.linalg.norm(diagnostic_coils, axis=1)
    assert np.isfinite(mean_b0_diagnostic_image).all(), "Invalid diagnostic reconstruction."
    show_table([
        ("Reliable ROI pixels used for mean B0", int(measured_roi.sum())),
        ("Measured mean B0 [Hz]", mean_b0_hz),
        ("B0 standard deviation in this ROI [Hz]", std_b0_hz),
        ("Predicted phase displacement [pixels]", predicted_shift_pixels),
        ("Hardware comparison uses", "Original scan_image; no offline B0 removal"),
    ])
    diagnostic_scale = max(scan_image.max(), mean_b0_diagnostic_image.max())
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
    image_panel(axes[0], scan_image, "Measured EPI: navigator correction",
                vmin=0, vmax=diagnostic_scale)
    image_panel(axes[1], mean_b0_diagnostic_image,
                f"Diagnostic: mean B0 removed ({mean_b0_hz:.1f} Hz)",
                vmin=0, vmax=diagnostic_scale)
    for ax in axes:
        ax.set(xlabel="Readout column", ylabel="Phase row")
    show_figure(fig)
else:
    print("Mean B0 diagnostic disabled, or the scan is GRE rather than EPI.")


## 9. ACDC response maps and MATLAB input

The original NPZ contains a 3D response volume in Hz/A. The confirmed patient-to-source
transform samples that volume at the simulation slice. The existing interpolation code
preserves channel order and leaves unknown responses as NaN. Show all 32 responses before
constructing the exact experiment struct consumed by the MATLAB functions.

In [ ]:
# Sample the measured calibration directly on the simulation grid.
geometry = {key: simulation_reference[key] for key in ("X_m", "Y_m", "position_mm", "spacing_mm", "orientation", "slice_thickness_mm")}
calibration = read_calibration(FILES["calibration"], {"geometry": geometry}, CALIBRATION_PARAMETERS)
coil_maps = calibration["acdc_maps"]
calibration_valid = calibration["spatial_valid_mask"]
fig, axes = plt.subplots(4, 8, figsize=(19, 10), layout="constrained")
scale = np.nanpercentile(np.abs(coil_maps), 99)
for channel, ax in enumerate(axes.ravel()):
    ax.imshow(coil_maps[..., channel], cmap="RdBu_r", vmin=-scale, vmax=scale, interpolation="nearest")
    ax.set_title(f"Channel {channel + 1}")
    ax.set_axis_off()
fig.suptitle(f"Measured ACDC responses on reference grid [Hz/A], common scale ±{scale:.1f}")
show_figure(fig)
print("Calibration shape:", coil_maps.shape, "; units:", calibration["units"])

In [ ]:
TRIM_TO_COMMON_VALID_SUPPORT = True
assert support_before_validity_filter.shape == calibration_valid.shape == b0_valid.shape, "ROI, ACDC, and B0 masks must share the simulation grid."
valid_acdc = calibration_valid & np.isfinite(coil_maps).all(axis=-1)
valid_b0 = b0_valid & np.isfinite(b0_on_reference)
common_valid = valid_acdc & valid_b0
acdc_invalid_pixels = support_before_validity_filter & ~valid_acdc
b0_invalid_pixels = support_before_validity_filter & ~valid_b0
invalid_for_either = acdc_invalid_pixels | b0_invalid_pixels
invalid_for_both = acdc_invalid_pixels & b0_invalid_pixels
removed_pixels = invalid_for_either.copy() if TRIM_TO_COMMON_VALID_SUPPORT else np.zeros_like(support_before_validity_filter, dtype=bool)
support = support_before_validity_filter & common_valid if TRIM_TO_COMMON_VALID_SUPPORT else support_before_validity_filter.copy()
excluded_object_pixels = threshold_support & ~support
excluded_signal_fraction = reference_magnitude[~support].sum() / reference_magnitude.sum()
ROI_PARAMETERS.update({
    "common_validity_filter_enabled": bool(TRIM_TO_COMMON_VALID_SUPPORT),
    "selected_pixels_invalid_for_acdc_calibration": int(acdc_invalid_pixels.sum()),
    "selected_pixels_invalid_for_b0": int(b0_invalid_pixels.sum()),
    "selected_pixels_invalid_for_both": int(invalid_for_both.sum()),
    "pixels_removed_for_invalid_acdc_or_b0": int(removed_pixels.sum()),
    "selected_pixels_after_common_validity_filter": int(support.sum()),
})
fig, axes = plt.subplots(1, 4, figsize=(16, 4), layout="constrained")
image_panel(axes[0], support_before_validity_filter, f"Original support: {int(support_before_validity_filter.sum())} pixels", vmin=0, vmax=1)
image_panel(axes[1], valid_acdc, "Valid ACDC responses", vmin=0, vmax=1)
image_panel(axes[2], valid_b0, "Finite, valid B0", vmin=0, vmax=1)
image_panel(axes[3], support, f"Common-valid support: {int(support.sum())} pixels", vmin=0, vmax=1)
show_figure(fig)
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
image_panel(ax, reference_magnitude, f"Pixels removed for invalid ACDC or B0: {int(removed_pixels.sum())}")
removed_rows, removed_columns = np.where(removed_pixels)
ax.scatter(removed_columns, removed_rows, marker="s", s=28, c="red")
show_figure(fig)
show_table([("Common-validity filter enabled", TRIM_TO_COMMON_VALID_SUPPORT),
            ("Pixels invalid for ACDC", int(acdc_invalid_pixels.sum())),
            ("Pixels invalid for B0", int(b0_invalid_pixels.sum())),
            ("Pixels invalid for both", int(invalid_for_both.sum())),
            ("Pixels removed (unique total)", int(removed_pixels.sum())),
            ("Selected pixels remaining", int(support.sum())),
            ("Reference signal excluded", f"{excluded_signal_fraction:.1%}")])
assert support.any(), "ACDC/B0 validity filtering removed the entire selected support."
assert np.all(common_valid[support]), "Selected support contains an invalid ACDC or B0 pixel; enable TRIM_TO_COMMON_VALID_SUPPORT."

In [ ]:
# Build the existing MATLAB struct transparently; no new forward model is implemented in Python.
scale = reference_magnitude[support].max()
m0_proxy = np.where(support, reference_magnitude / scale, 0.0)
source_hashes = {name: hashlib.sha256(path.read_bytes()).hexdigest() for name, path in FILES.items()}
experiment = {
    "images": {"gre_reference": reference_magnitude, "scan": scan_image, "perturbed": scan_image},
    "field": {"b0Hz": np.where(support, b0_on_reference, 0.0),
              "measured_b0Hz": np.where(b0_valid, b0_on_reference, np.nan),
              "mask": b0_valid, "simulation_mask": support},
    "geometry": geometry, "acquisition": acquisition, "m0_proxy": m0_proxy,
    "relaxation": {"T2star_s": T2STAR_S},
    "current_response": CURRENT_RESPONSE,
    "assumptions": np.array([
        "GRE magnitude is a transverse-amplitude proxy; reference/scan contrast is not calibrated.",
        "Only the explicitly chosen ROI is simulated; every ROI field value is measured.",
        "Uniform receive sensitivity; ideal GRE spoiling.",
        "ACDC: first-order temporal response when enabled; zero initial current; experimental validation required.",
        "Component frequency branches and scanner B0 sign need independent verification.",
    ], dtype=object),
    "provenance": {"source_sha256": source_hashes, "B0_sign": B0_SIGN,
                   "GRE_amplitude_normalization": float(scale), "roi_parameters": ROI_PARAMETERS,
                   "simulation_grid": simulation_grid_report,
                   "navigator_phase_branch": NAVIGATOR_PHASE_BRANCH,
                   "selected_roi_sha256": hashlib.sha256(np.ascontiguousarray(support, dtype=np.uint8).tobytes()).hexdigest()},
    "optimization_ready": True,
}
check_calibration(calibration, experiment)
INPUT_MAT = WORK_DIR / "experiment_input.mat"
CALIBRATION_MAT = WORK_DIR / "calibration.mat"
savemat(INPUT_MAT, {"experiment": experiment}, do_compression=True, long_field_names=True)
savemat(CALIBRATION_MAT, calibration, do_compression=True, long_field_names=True)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
image_panel(axes[0], m0_proxy, "Normalized MATLAB object amplitude")
plot = image_panel(axes[1], np.where(support, b0_on_reference, np.nan), "Measured field used by MATLAB [Hz]", cmap="RdBu_r")
fig.colorbar(plot, ax=axes[1])
show_figure(fig)
print("Internal MATLAB inputs:", INPUT_MAT, CALIBRATION_MAT)

## 9b. Static B0 shimming with Haotian's solver

Run through sections 9b and 9c to obtain the currents for the first, constant-current scan.
This step reuses `solve_shim_static_qp` from
`ACDC_shim_static_dyn_haotian/ACDC_Optim/ACDC_optimization.py`.
It uses the same selected ROI, measured B0 and registered calibration as the dynamic branch.
The original static solver is Python; the existing forward model and dynamic optimization
below remain MATLAB.

The solver fits a **target field**. We pass the negative measured B0 so that
`B0 + ACDC` approaches zero, with one constant current per channel:

$$\min_I \|B_0 + W^T I\|_2^2,\qquad
|I_c|\leq I_{max},\qquad \sum_c |I_c| < I_{total}.$$

Small numerical margins keep the solution inside the strict export limits.
The QP uses current bounds; the existing `FIELD_LIMIT_HZ` is checked before export.
The field RMS and NRMSE shown here describe the **settled field**, rather than image error.
The example notebook's synthetic target and spherical ROI are replaced with our measured
field and the ROI already verified above.


In [ ]:
# Static-only parameters: leave the dynamic experiment and its parameters unchanged.
RUN_STATIC_SHIM = True
EXPORT_STATIC_CURRENTS = True
STATIC_MAX_CURRENT_A = MAX_CURRENT_A
STATIC_TOTAL_CURRENT_LIMIT_A = TOTAL_CURRENT_LIMIT_A
STATIC_CSV_LAST_TIME_S = 1.0  # Seconds after the first readout trigger; cover the full scan duration.
STATIC_OUTPUT_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + "_static")

static_currents_A = None
static_export_ready = False
static_csv_file = None
if RUN_STATIC_SHIM:
    sys.path.insert(0, str(EXPERIMENT_DIR / "ACDC_shim_static_dyn_haotian"))
    from ACDC_Optim import solve_shim_static_qp

    assert np.isscalar(STATIC_MAX_CURRENT_A) and np.isfinite(STATIC_MAX_CURRENT_A) and STATIC_MAX_CURRENT_A > 1e-8, "Use a positive scalar static current limit."
    assert np.isscalar(STATIC_TOTAL_CURRENT_LIMIT_A) and np.isfinite(STATIC_TOTAL_CURRENT_LIMIT_A) and STATIC_TOTAL_CURRENT_LIMIT_A > 1e-4, "Use a positive static total-current limit."
    static_mask = support.copy()
    static_measured_b0_Hz = np.asarray(b0_on_reference[static_mask], dtype=float)
    static_basis_Hz_per_A = np.asarray(coil_maps[static_mask], dtype=float).T
    assert static_measured_b0_Hz.size > 0 and np.isfinite(static_measured_b0_Hz).all(), "The static ROI needs finite measured B0 values."
    assert static_basis_Hz_per_A.shape == (32, static_measured_b0_Hz.size) and np.isfinite(static_basis_Hz_per_A).all(), "The static ROI needs all 32 calibrated channels."

    if np.any(static_measured_b0_Hz != 0):
        static_currents_A, static_fit_nrmse = solve_shim_static_qp(
            -static_measured_b0_Hz, static_basis_Hz_per_A,
            amp_limit=STATIC_MAX_CURRENT_A - 1e-8,
            l1_limit=STATIC_TOTAL_CURRENT_LIMIT_A - 1e-4,
        )
    else:
        static_currents_A = np.zeros(32)
        static_fit_nrmse = None  # Relative error is undefined for a zero reference field.

    static_generated_field_Hz = coil_maps @ static_currents_A
    static_residual_field_Hz = b0_on_reference + static_generated_field_Hz
    static_quantized_currents_A = quantize_currents(static_currents_A)
    static_quantized_field_Hz = coil_maps @ static_quantized_currents_A
    static_quantized_residual_Hz = b0_on_reference + static_quantized_field_Hz
    static_before_rms_Hz = float(np.sqrt(np.mean(static_measured_b0_Hz**2)))
    static_after_rms_Hz = float(np.sqrt(np.mean(static_residual_field_Hz[static_mask]**2)))
    static_quantized_rms_Hz = float(np.sqrt(np.mean(static_quantized_residual_Hz[static_mask]**2)))
    static_checks = [
        ("Finite static currents", np.isfinite(static_currents_A).all()),
        ("Static per-channel current limit", np.all(np.abs(static_currents_A) <= STATIC_MAX_CURRENT_A)),
        ("Static total-current limit", np.abs(static_currents_A).sum() < STATIC_TOTAL_CURRENT_LIMIT_A),
        ("Static generated-field limit on ROI", np.max(np.abs(static_generated_field_Hz[static_mask])) <= FIELD_LIMIT_HZ + 1e-8),
        ("Quantized current limits", np.all(np.abs(static_quantized_currents_A) <= STATIC_MAX_CURRENT_A) and np.abs(static_quantized_currents_A).sum() < STATIC_TOTAL_CURRENT_LIMIT_A),
        ("Quantized generated-field limit on ROI", np.max(np.abs(static_quantized_field_Hz[static_mask])) <= FIELD_LIMIT_HZ + 1e-8),
        ("Predicted field RMS does not increase", static_after_rms_Hz <= static_before_rms_Hz + 1e-8),
    ]
    static_export_ready = all(passed for _, passed in static_checks)
    show_table([(name, "PASS" if passed else "FAIL") for name, passed in static_checks], ("Static result check", "Result"))
    show_table([
        ("Static ROI pixels", int(static_mask.sum())),
        ("Field RMS before [Hz]", static_before_rms_Hz),
        ("Predicted settled field RMS after [Hz]", static_after_rms_Hz),
        ("Nominal DAC-quantized field RMS after [Hz]", static_quantized_rms_Hz),
        ("Field standard deviation before [Hz]", float(np.std(static_measured_b0_Hz))),
        ("Predicted field standard deviation after [Hz]", float(np.std(static_residual_field_Hz[static_mask]))),
        ("Field NRMSE after [%]", None if static_fit_nrmse is None else float(static_fit_nrmse * 100)),
        ("Maximum static channel current [A]", float(np.abs(static_currents_A).max())),
        ("Sum of absolute static currents [A]", float(np.abs(static_currents_A).sum())),
    ])
else:
    print("Static shimming disabled in this section's parameter cell.")


### 9c. Inspect static currents and export the constant-current CSV

The CSV has **two headerless rows**: time in seconds, then physical-output currents
in amperes. Both rows contain the same current vector, using the existing channel
mapping and polarity. With the default 32 outputs, the file has **33 columns**:

```text
0,I1,I2,...,I32
1,I1,I2,...,I32
```

Your first imaging-readout trigger starts this constant-current scan. The first command
is applied at that trigger and settles according to the channel response; phase accumulated
before the trigger remains. Keep the currents applied through this scan, then switch them
off before using the subsequent independent dynamic waveforms. Choose
`STATIC_CSV_LAST_TIME_S` to cover the scan duration after the trigger; controller playback
and power-off remain external to this notebook. The CSV has no terminal zero-current row.

The static file is saved in a **separate sibling folder** ending in `_static`, so the
existing dynamic output folder still receives only `waveform.csv`. This step keeps the
measured B0, MATLAB input, dynamic currents and dynamic export unchanged.


In [ ]:
# Inspect the steady-state field and the 32 currents before creating the static CSV.
if static_currents_A is not None:
    static_plot_scale_Hz = max(1.0, float(np.max(np.abs(np.r_[
        static_measured_b0_Hz, static_generated_field_Hz[static_mask], static_residual_field_Hz[static_mask]]))))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
    for ax, values, title in zip(axes,
        (b0_on_reference, static_generated_field_Hz, static_residual_field_Hz),
        ("Measured B0 before static [Hz]", "Generated steady-state ACDC field [Hz]", "Predicted settled B0 + static [Hz]")):
        plot = image_panel(ax, np.where(static_mask, values, np.nan), title, cmap="RdBu_r",
                           vmin=-static_plot_scale_Hz, vmax=static_plot_scale_Hz)
        fig.colorbar(plot, ax=ax)
    show_figure(fig)

    fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
    ax.bar(np.arange(1, 33), static_currents_A)
    ax.axhline(STATIC_MAX_CURRENT_A, color="red", linestyle="--", label="Channel current limits")
    ax.axhline(-STATIC_MAX_CURRENT_A, color="red", linestyle="--")
    ax.set(xlabel="Calibration channel", ylabel="Static DAC command [A]", title="One constant current per ACDC channel")
    ax.set_xticks(np.arange(1, 33))
    ax.legend()
    show_figure(fig)

    from export_acdc_static import export_static_currents
    static_settings = dict(WAVEFORM_PARAMETERS, max_current_A=STATIC_MAX_CURRENT_A,
                           total_current_limit_A=STATIC_TOTAL_CURRENT_LIMIT_A)
    static_output_numbers = np.asarray(static_settings["channel_map"])
    static_polarity = np.asarray(static_settings["channel_polarity"])
    show_table([(channel + 1, int(static_output_numbers[channel]), f"{static_currents_A[channel]:+.6f}",
                 f"{static_currents_A[channel] * static_polarity[channel]:+.6f}") for channel in range(32)],
               ("Calibration channel", "Physical output (1-based)", "Model current [A]", "System current [A]"))

    if EXPORT_STATIC_CURRENTS and static_export_ready:
        static_export = export_static_currents(static_currents_A, static_settings, STATIC_OUTPUT_DIR,
                                               hold_timestamp_s=STATIC_CSV_LAST_TIME_S)
        static_csv_file = Path(static_export["csv_file"])
        static_csv_data = np.loadtxt(static_csv_file, delimiter=",")
        assert static_csv_data.shape == (2, static_export["physical_channels"] + 1)
        assert np.array_equal(static_csv_data[0, 1:], static_csv_data[1, 1:]), "Both static CSV rows must have identical currents."
        fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
        ax.plot(static_csv_data[:, 0], static_csv_data[:, 1:], alpha=0.6)
        ax.set(xlabel="Time after first readout trigger [s]", ylabel="Physical output current [A]",
               title="Static CSV commands: identical currents in both rows")
        show_figure(fig)
        print("Static currents CSV:", static_csv_file)
        print("The following MATLAB dynamic pipeline uses its existing independent inputs.")
    else:
        print("Static CSV export disabled or static numerical checks failed.")


## 9d. ACDC Dynamic B₀ Shimming: target and temporal response

This independent branch reuses Haotian's `solve_shim_waveform_fista` in NumPy.
It fits the **field**, using one DAC command per imaging line, with the same ROI,
measured B0 and registered calibration as the static step. The target is `-B0`
at every imaging ADC sample. We have one measured map, so B0 is assumed constant
throughout this scan. Waveform variation compensates the channel response, mainly
at startup; no time-varying B0 or eddy-current field is inferred.

The first imaging-readout trigger starts this waveform after the navigators.
`B0_DYNAMIC_TRIGGER_DELAY_S` locates that trigger relative to the **beginning of the
first imaging-readout block**, in seconds. The default is zero; if the trigger is
at the ADC start, enter the ADC delay instead. The timing table below shows both
positions. This branch's time origin is independent of the later MATLAB export.

Commands are held over the sequence's imaging windows. Actual currents start at
zero and retain their history. For each channel, the exact response matrix is

$$H_{ij}=G(t_i-s_j)-G(t_i-e_j),\qquad
G(d)=1-\exp[-\max(d,0)/\tau].$$

`CURRENT_RESPONSE_ENABLED` and `CURRENT_RESPONSE_TAU_US` also apply here, including
32 different channel time constants. With the response disabled, H is the ideal
piecewise-constant command operator. FISTA minimizes the field residual at **every imaging ADC**, while holding one
command per imaging line. We also check the predicted currents and fields there.
This does not optimize accumulated phase or restore phase acquired before the
trigger. The existing MATLAB forward model and image optimization continue below.


In [ ]:
# Dynamic B0 field fitting: independent parameters and a separate CSV destination.
RUN_DYNAMIC_B0_SHIM = True
EXPORT_DYNAMIC_B0_WAVEFORM = True
B0_DYNAMIC_MAX_ITERATIONS = 2000
B0_DYNAMIC_RIDGE_X = 1e-6  # Haotian's regularization after spectral scaling of the fields.
B0_DYNAMIC_TRIGGER_DELAY_S = 0.0  # Seconds after the first imaging-readout block begins.
B0_DYNAMIC_OUTPUT_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + "_dynamic_b0")


def first_order_response_matrix(sample_times_s, state_starts_s, state_ends_s, tau_s, enabled=True):
    """Map held DAC commands to actual currents: [channel, sample, state]."""
    samples = np.asarray(sample_times_s, dtype=float)
    state_starts = np.asarray(state_starts_s, dtype=float)
    state_ends = np.asarray(state_ends_s, dtype=float)
    tau = np.atleast_1d(np.asarray(tau_s, dtype=float))
    assert samples.ndim == 1 and samples.size and np.isfinite(samples).all() and (samples >= 0).all(), "Use finite nonnegative sample times."
    assert state_starts.ndim == 1 and state_starts.size and state_ends.shape == state_starts.shape, "Each state needs one start and one end."
    assert np.isfinite(state_starts).all() and np.isfinite(state_ends).all() and (state_starts >= 0).all(), "Use finite nonnegative state windows."
    assert np.all(state_ends > state_starts) and np.all(np.diff(state_starts) > 0) and np.all(state_ends[:-1] <= state_starts[1:] + 1e-12), "State windows must be ordered and nonoverlapping."
    assert tau.ndim == 1 and tau.size and np.isfinite(tau).all() and (tau > 0).all(), "Use positive finite channel time constants."
    assert isinstance(enabled, (bool, np.bool_)), "Use True or False for the temporal response."
    if not enabled:
        ideal = (samples[:, None] >= state_starts) & (samples[:, None] < state_ends)
        return np.broadcast_to(ideal, (len(tau), len(samples), len(state_starts))).astype(float)
    age_start = np.maximum(samples[:, None] - state_starts, 0.0)
    age_end = np.maximum(samples[:, None] - state_ends, 0.0)
    step_start = -np.expm1(-age_start[None, :, :] / tau[:, None, None])
    step_end = -np.expm1(-age_end[None, :, :] / tau[:, None, None])
    return step_start - step_end


b0_dynamic_target_Hz = None
b0_dynamic_currents_A = None
b0_dynamic_export_ready = False
b0_dynamic_csv_file = None
if RUN_DYNAMIC_B0_SHIM:
    assert timing["sequence_kind"] == "epi", "This dynamic B0 export currently uses contiguous single-shot EPI imaging windows."
    assert np.allclose(ends[:-1], starts[1:], atol=1e-12, rtol=0), "Imaging windows must be contiguous."
    assert 0 < MAX_CURRENT_A <= 3 and TOTAL_CURRENT_LIMIT_A > 1e-4, "Use scalar positive current limits, at most 3 A per channel."
    assert isinstance(B0_DYNAMIC_MAX_ITERATIONS, int) and B0_DYNAMIC_MAX_ITERATIONS > 0, "Use a positive integer iteration limit."
    assert np.isscalar(B0_DYNAMIC_RIDGE_X) and np.isfinite(B0_DYNAMIC_RIDGE_X) and B0_DYNAMIC_RIDGE_X >= 0, "Use a finite nonnegative ridge weight."
    assert np.isscalar(B0_DYNAMIC_TRIGGER_DELAY_S) and np.isfinite(B0_DYNAMIC_TRIGGER_DELAY_S) and B0_DYNAMIC_TRIGGER_DELAY_S >= 0, "Enter the nonnegative trigger delay in seconds."

    b0_dynamic_mask = support.copy()
    b0_dynamic_basis = np.asarray(coil_maps[b0_dynamic_mask], dtype=float).T
    b0_dynamic_trigger_rf_s = float(starts[0] + B0_DYNAMIC_TRIGGER_DELAY_S)
    b0_dynamic_first_adc_rf_s = float(np.min(acquisition["time_s"]))
    assert b0_dynamic_trigger_rf_s <= b0_dynamic_first_adc_rf_s, "The trigger must occur by the first imaging ADC sample."
    b0_dynamic_edges_s = np.r_[0.0, starts[1:] - b0_dynamic_trigger_rf_s, ends[-1] - b0_dynamic_trigger_rf_s]
    assert np.all(np.diff(b0_dynamic_edges_s) > 0), "The trigger must leave positive imaging state durations."
    b0_dynamic_line_center_s = np.array([
        np.mean(acquisition["time_s"][acquisition["line_index"] == line + 1])
        for line in range(len(starts))]) - b0_dynamic_trigger_rf_s
    b0_dynamic_adc_s = np.sort(np.asarray(acquisition["time_s"]).ravel()) - b0_dynamic_trigger_rf_s
    b0_dynamic_target_Hz = np.broadcast_to(-b0_on_reference[b0_dynamic_mask],
                                          (len(b0_dynamic_adc_s), int(b0_dynamic_mask.sum()))).copy()
    assert np.isfinite(b0_dynamic_target_Hz).all() and np.isfinite(b0_dynamic_basis).all(), "The selected ROI needs finite B0 and calibration values."
    b0_dynamic_response = first_order_response_matrix(
        b0_dynamic_adc_s, b0_dynamic_edges_s[:-1], b0_dynamic_edges_s[1:],
        CURRENT_RESPONSE["tau_s"], enabled=CURRENT_RESPONSE_ENABLED)
    b0_dynamic_line_response = first_order_response_matrix(
        b0_dynamic_line_center_s, b0_dynamic_edges_s[:-1], b0_dynamic_edges_s[1:],
        CURRENT_RESPONSE["tau_s"], enabled=CURRENT_RESPONSE_ENABLED)
    show_table([
        ("Target", "Negative measured B0, repeated at every imaging ADC sample"),
        ("Imaging command states / calibrated channels", (len(starts), b0_dynamic_basis.shape[0])),
        ("Imaging ADC samples used in the field objective", len(b0_dynamic_adc_s)),
        ("First imaging block from RF [ms]", float(starts[0] * 1000)),
        ("First imaging ADC sample from RF [ms]", b0_dynamic_first_adc_rf_s * 1000),
        ("ADC start delay within first readout block [us]", (b0_dynamic_first_adc_rf_s - starts[0] - acquisition["dwell_s"] / 2) * 1e6),
        ("Selected trigger from RF [ms]", b0_dynamic_trigger_rf_s * 1000),
        ("Last CSV timestamp after trigger [ms]", b0_dynamic_edges_s[-1] * 1000),
        ("Temporal response enabled", CURRENT_RESPONSE_ENABLED),
        ("Channel tau range [us]", [float(tau_us.min()), float(tau_us.max())]),
    ])
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
    plot = image_panel(axes[0], np.where(b0_dynamic_mask, -b0_on_reference, np.nan),
                       "Target ACDC field at every imaging ADC [Hz]", cmap="RdBu_r")
    fig.colorbar(plot, ax=axes[0])
    plot = axes[1].imshow(b0_dynamic_response[0], origin="upper", cmap="viridis", aspect="auto")
    axes[1].set(xlabel="Command state", ylabel="Chronological imaging ADC sample", title="Exact temporal response: calibration channel 1")
    fig.colorbar(plot, ax=axes[1], label="Actual-current weight")
    show_figure(fig)
else:
    print("Dynamic B0 shimming disabled in this section's parameter cell.")


### 9e. Optimize the dynamic B0 waveform with Haotian's FISTA solver

The solver returns `[state, channel]` **DAC commands**, not the filtered currents.
It uses the same per-channel and total-current limits, with small numerical margins.
The exact response matrices map imaging-line commands to every ADC sample, replacing
the example's uniformly sampled SIRF. The sequence's actual timing and channel-specific
tau are respected, including the beginning of the first readout.

`B0_DYNAMIC_RIDGE_X` is the regularization weight on the spectrally scaled problem,
as in Haotian's implementation. The iteration limit is specific to this branch.
Inspect the convergence plot and the recomputed field errors; this is a field-fitting
objective, independent of the later MATLAB image MSE.


In [ ]:
# Reuse the existing NumPy FISTA method; no MATLAB image optimization is run here.
if b0_dynamic_target_Hz is not None:
    sys.path.insert(0, str(EXPERIMENT_DIR / "ACDC_shim_static_dyn_haotian"))
    from ACDC_Optim import solve_shim_waveform_fista

    if np.any(b0_dynamic_target_Hz != 0):
        b0_dynamic_currents_A, b0_dynamic_fit_nrmse, b0_dynamic_loss = solve_shim_waveform_fista(
            b0_dynamic_target_Hz, b0_dynamic_basis, conv_matrix=b0_dynamic_response,
            amp_limit=MAX_CURRENT_A - 1e-8, l1_limit=TOTAL_CURRENT_LIMIT_A - 1e-4,
            ridge_x=B0_DYNAMIC_RIDGE_X, max_iter=B0_DYNAMIC_MAX_ITERATIONS,
            tol=1e-10, verbose=False, plot=True,
        )
    else:
        b0_dynamic_currents_A = np.zeros((len(starts), 32))
        b0_dynamic_fit_nrmse = None  # Relative error is undefined for a zero target.
        b0_dynamic_loss = 0.0

    b0_dynamic_actual_line_A = np.einsum("cls,sc->lc", b0_dynamic_line_response, b0_dynamic_currents_A)
    b0_dynamic_generated_line_Hz = b0_dynamic_actual_line_A @ b0_dynamic_basis
    b0_dynamic_residual_line_Hz = b0_dynamic_generated_line_Hz + b0_on_reference[b0_dynamic_mask]
    b0_dynamic_adc_response = b0_dynamic_response
    b0_dynamic_actual_adc_A = np.einsum("cts,sc->tc", b0_dynamic_adc_response, b0_dynamic_currents_A)
    b0_dynamic_generated_adc_Hz = b0_dynamic_actual_adc_A @ b0_dynamic_basis
    b0_dynamic_residual_adc_Hz = b0_dynamic_generated_adc_Hz + b0_on_reference[b0_dynamic_mask]
    b0_dynamic_quantized_A = quantize_currents(b0_dynamic_currents_A)
    b0_dynamic_quantized_actual_adc_A = np.einsum("cts,sc->tc", b0_dynamic_adc_response, b0_dynamic_quantized_A)
    b0_dynamic_quantized_adc_Hz = b0_dynamic_quantized_actual_adc_A @ b0_dynamic_basis
    b0_dynamic_commanded_field_Hz = b0_dynamic_currents_A @ b0_dynamic_basis
    b0_dynamic_quantized_commanded_field_Hz = b0_dynamic_quantized_A @ b0_dynamic_basis
    b0_dynamic_before_rms_Hz = float(np.sqrt(np.mean(b0_dynamic_target_Hz**2)))
    b0_dynamic_after_rms_Hz = float(np.sqrt(np.mean(b0_dynamic_residual_adc_Hz**2)))
    b0_dynamic_line_center_rms_Hz = float(np.sqrt(np.mean(b0_dynamic_residual_line_Hz**2)))
    b0_dynamic_adc_rms_Hz = np.sqrt(np.mean(b0_dynamic_residual_adc_Hz**2, axis=1))
    b0_dynamic_checks = [
        ("Finite current commands with one state per imaging line", b0_dynamic_currents_A.shape == (len(starts), 32) and np.isfinite(b0_dynamic_currents_A).all()),
        ("DAC command current limits", np.all(np.abs(b0_dynamic_currents_A) <= MAX_CURRENT_A) and np.all(np.abs(b0_dynamic_currents_A).sum(axis=1) < TOTAL_CURRENT_LIMIT_A)),
        ("Generated command-field limit on ROI", np.max(np.abs(b0_dynamic_commanded_field_Hz)) <= FIELD_LIMIT_HZ + 1e-8),
        ("Actual-current limits at every imaging ADC", np.all(np.abs(b0_dynamic_actual_adc_A) <= MAX_CURRENT_A + 1e-8) and np.all(np.abs(b0_dynamic_actual_adc_A).sum(axis=1) < TOTAL_CURRENT_LIMIT_A)),
        ("Actual generated-field limit at every imaging ADC", np.max(np.abs(b0_dynamic_generated_adc_Hz)) <= FIELD_LIMIT_HZ + 1e-8),
        ("Quantized command current limits", np.all(np.abs(b0_dynamic_quantized_A) <= MAX_CURRENT_A) and np.all(np.abs(b0_dynamic_quantized_A).sum(axis=1) < TOTAL_CURRENT_LIMIT_A)),
        ("Quantized command-field limit on ROI", np.max(np.abs(b0_dynamic_quantized_commanded_field_Hz)) <= FIELD_LIMIT_HZ + 1e-8),
        ("Quantized actual-current limits at every imaging ADC", np.all(np.abs(b0_dynamic_quantized_actual_adc_A) <= MAX_CURRENT_A + 1e-8) and np.all(np.abs(b0_dynamic_quantized_actual_adc_A).sum(axis=1) < TOTAL_CURRENT_LIMIT_A)),
        ("Quantized generated-field limit at every imaging ADC", np.max(np.abs(b0_dynamic_quantized_adc_Hz)) <= FIELD_LIMIT_HZ + 1e-8),
        ("Predicted all-ADC field RMS does not increase", b0_dynamic_after_rms_Hz <= b0_dynamic_before_rms_Hz + 1e-8),
    ]
    b0_dynamic_export_ready = all(passed for _, passed in b0_dynamic_checks)
    show_table([(name, "PASS" if passed else "FAIL") for name, passed in b0_dynamic_checks], ("Dynamic B0 result check", "Result"))
    show_table([
        ("Field RMS before [Hz]", b0_dynamic_before_rms_Hz),
        ("Predicted field RMS at line centers [Hz]", b0_dynamic_line_center_rms_Hz),
        ("Predicted field RMS over all imaging ADCs [Hz]", float(np.sqrt(np.mean(b0_dynamic_residual_adc_Hz**2)))),
        ("Worst imaging ADC field RMS [Hz]", float(b0_dynamic_adc_rms_Hz.max())),
        ("All-ADC field NRMSE [%]", None if b0_dynamic_fit_nrmse is None else float(b0_dynamic_fit_nrmse * 100)),
        ("Maximum absolute DAC command [A]", float(np.abs(b0_dynamic_currents_A).max())),
        ("Maximum total absolute DAC current [A]", float(np.abs(b0_dynamic_currents_A).sum(axis=1).max())),
        ("Final scaled FISTA objective", float(b0_dynamic_loss)),
    ])
else:
    print("Dynamic B0 optimization skipped because this branch is disabled.")


### 9f. Inspect the predicted dynamic waveform and export its CSV

The plots show the field at the first, middle and last line centers, the commanded
and predicted actual currents, and the residual-field RMS at every imaging ADC.
The full current preview includes the decay after the final zero command.

Only **DAC commands** are exported, using the existing channel mapping and polarity.
The CSV starts at the selected first-imaging-readout trigger, uses one state per
imaging line, and has a final zero-current row at the end of imaging. It is saved
as `waveform.csv` in a separate sibling folder ending in `_dynamic_b0`.
The static CSV and the later MATLAB image-optimization CSV remain independent.
No FPGA operation is performed.


In [ ]:
# Plot actual currents from the complete command history, including the final decay.
if b0_dynamic_currents_A is not None:
    b0_dynamic_plot_times_s = np.unique(np.r_[
        b0_dynamic_edges_s, b0_dynamic_line_center_s,
        np.linspace(0, b0_dynamic_edges_s[-1], 500),
        (b0_dynamic_edges_s[:, None] + np.linspace(0, 8 * float(tau_us.max()) * 1e-6, 81)).ravel(),
    ])
    b0_dynamic_plot_response = first_order_response_matrix(
        b0_dynamic_plot_times_s, b0_dynamic_edges_s[:-1], b0_dynamic_edges_s[1:],
        CURRENT_RESPONSE["tau_s"], enabled=CURRENT_RESPONSE_ENABLED)
    b0_dynamic_plot_actual_A = np.einsum("cts,sc->tc", b0_dynamic_plot_response, b0_dynamic_currents_A)
    b0_dynamic_line_snapshots = sorted(set([0, len(starts) // 2, len(starts) - 1]))
    b0_dynamic_field_scale_Hz = max(1.0, float(np.max(np.abs(b0_dynamic_target_Hz))))
    fig, axes = plt.subplots(2, len(b0_dynamic_line_snapshots), figsize=(15, 8), layout="constrained", squeeze=False)
    for column, line in enumerate(b0_dynamic_line_snapshots):
        for row, values, name in (
            (0, b0_dynamic_generated_line_Hz[line], "Actual ACDC field"),
            (1, b0_dynamic_residual_line_Hz[line], "Residual B0 + ACDC")):
            b0_dynamic_panel = np.full(support.shape, np.nan)
            b0_dynamic_panel[b0_dynamic_mask] = values
            plot = image_panel(axes[row, column], b0_dynamic_panel,
                f"{name}, line {line + 1} [Hz]", cmap="RdBu_r",
                vmin=-b0_dynamic_field_scale_Hz, vmax=b0_dynamic_field_scale_Hz)
            fig.colorbar(plot, ax=axes[row, column])
    show_figure(fig)

    fig, axes = plt.subplots(1, 3, figsize=(17, 4), layout="constrained")
    axes[0].step(b0_dynamic_edges_s * 1000, np.vstack([b0_dynamic_currents_A, np.zeros((1, 32))]),
                 where="post", alpha=0.4)
    axes[0].plot(b0_dynamic_plot_times_s * 1000, b0_dynamic_plot_actual_A, alpha=0.5)
    axes[0].set(xlabel="Time after trigger [ms]", ylabel="Calibration current [A]", title="32 channels: DAC steps and predicted actual currents")
    axes[1].plot(b0_dynamic_plot_times_s * 1000, np.abs(b0_dynamic_plot_actual_A).sum(axis=1), label="Predicted actual")
    axes[1].step(b0_dynamic_edges_s * 1000, np.r_[np.abs(b0_dynamic_currents_A).sum(axis=1), 0], where="post", label="DAC commands")
    axes[1].set(xlabel="Time after trigger [ms]", ylabel="Sum of absolute currents [A]", title="Total current through imaging and final decay")
    axes[1].legend()
    axes[2].plot(b0_dynamic_adc_s * 1000, b0_dynamic_adc_rms_Hz, label="Predicted dynamic B0 residual")
    axes[2].axhline(b0_dynamic_before_rms_Hz, color="gray", linestyle="--", label="Before ACDC")
    axes[2].set(xlabel="Time after trigger [ms]", ylabel="Field RMS on ROI [Hz]", title="Field error at every imaging ADC")
    axes[2].legend()
    show_figure(fig)

    # Zoom on startup: one automatically selected channel, using exact actual-current samples.
    b0_dynamic_preview_channel = int(np.argmax(np.max(np.abs(b0_dynamic_currents_A), axis=0)))
    fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
    ax.step(b0_dynamic_edges_s * 1e6, np.r_[b0_dynamic_currents_A[:, b0_dynamic_preview_channel], 0],
            where="post", label="DAC command")
    ax.plot(b0_dynamic_plot_times_s * 1e6, b0_dynamic_plot_actual_A[:, b0_dynamic_preview_channel], label="Predicted actual current")
    ax.axvline(float(b0_dynamic_adc_s[0] * 1e6), color="gray", linestyle="--", label="First imaging ADC sample")
    ax.set(xlim=(0, min(float(b0_dynamic_edges_s[-1] * 1e6), max(8 * float(tau_us.max()), float(b0_dynamic_adc_s[0] * 1e6) + 20))),
           xlabel="Time after trigger [us]", ylabel="Current [A]",
           title=f"Startup response: calibration channel {b0_dynamic_preview_channel + 1}")
    ax.legend()
    show_figure(fig)

    if EXPORT_DYNAMIC_B0_WAVEFORM and b0_dynamic_export_ready:
        b0_dynamic_export = export_waveform(b0_dynamic_currents_A.T, b0_dynamic_edges_s,
                                           WAVEFORM_PARAMETERS, B0_DYNAMIC_OUTPUT_DIR)
        b0_dynamic_csv_file = Path(b0_dynamic_export["csv_file"])
        b0_dynamic_csv = np.loadtxt(b0_dynamic_csv_file, delimiter=",")
        assert b0_dynamic_csv.shape == (len(starts) + 1, b0_dynamic_export["physical_channels"] + 1)
        assert b0_dynamic_csv[0, 0] == 0 and np.all(b0_dynamic_csv[-1, 1:] == 0)
        show_table([
            ("Dynamic B0 CSV", b0_dynamic_csv_file),
            ("Current states / CSV data rows", (len(starts), len(b0_dynamic_csv))),
            ("Last zero-current timestamp [ms]", b0_dynamic_csv[-1, 0] * 1000),
        ])
        print("Dynamic B0 waveform CSV:", b0_dynamic_csv_file)
        print("The following MATLAB image optimization uses its existing independent inputs.")
    else:
        print("Dynamic B0 export disabled or numerical checks failed.")
else:
    print("Dynamic B0 waveform preview skipped because this branch is disabled.")


## 9g. Choose the image-based optimizer's initial currents

`IMAGE_INITIALIZATION` in the main parameter cell selects the MATLAB solver's
**starting DAC commands**:

- `"static"`: repeat the validated `static_currents_A` from section 9b for every
  imaging state. This is the default.
- `"zero"`: use zero commands for every channel and every state.

This step uses the **static B0 shim result**, rather than the separate dynamic B0
waveform. No scaling, clipping or fallback is applied. Static initialization requires
the static calculation above; it does not require exporting its CSV.

The starting commands are passed directly to `fmincon` as its initial candidate.
This changes the numerical starting point, not the physical initial-current condition:
the temporal model still starts with zero actual current before the first command.
The measured B0 and the existing phase model are unchanged. The field figure below
is a settled-field preview, not an image prediction.

After changing this choice, restart execution from the first cell to create a new
working directory and regenerate the MATLAB request. The solver results save the
requested initial commands and their image MSE, which are checked in section 11.


In [ ]:
# Select an explicit starting point for the MATLAB image optimization.
assert IMAGE_INITIALIZATION in ("static", "zero"), 'IMAGE_INITIALIZATION must be "static" or "zero".'
if IMAGE_INITIALIZATION == "static":
    assert static_currents_A is not None and static_export_ready, "Run the static calculation in section 9b and resolve its checks, or select zero initialization."
    image_initial_currents_A = np.repeat(np.asarray(static_currents_A, dtype=float)[:, None], len(starts), axis=1)
else:
    image_initial_currents_A = np.zeros((32, len(starts)))

image_initial_generated_roi_Hz = coil_maps[support] @ image_initial_currents_A
image_initial_checks = [
    ("One command per calibrated channel and imaging state", image_initial_currents_A.shape == (32, len(starts))),
    ("Finite initial DAC commands", np.isfinite(image_initial_currents_A).all()),
    ("Initial per-channel current limit", np.all(np.abs(image_initial_currents_A) <= MAX_CURRENT_A)),
    ("Initial total-current limit", np.all(np.abs(image_initial_currents_A).sum(axis=0) < TOTAL_CURRENT_LIMIT_A)),
    ("Initial generated-field limit on ROI", np.max(np.abs(image_initial_generated_roi_Hz)) <= FIELD_LIMIT_HZ),
]
show_table([(name, "PASS" if passed else "FAIL") for name, passed in image_initial_checks], ("Image initialization check", "Result"))
assert all(passed for _, passed in image_initial_checks), "The selected starting point exceeds the image optimizer's limits; change the explicit settings or select zero initialization."
show_table([
    ("Image optimizer initialization", IMAGE_INITIALIZATION),
    ("Initial command array [channels, states]", image_initial_currents_A.shape),
    ("Maximum initial command [A]", float(np.abs(image_initial_currents_A).max())),
    ("Maximum initial sum of absolute currents [A]", float(np.abs(image_initial_currents_A).sum(axis=0).max())),
    ("Physical current before the first command [A]", 0.0),
])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
image_initial_color_limit_A = max(0.01, float(np.abs(image_initial_currents_A).max()))
plot = axes[0].imshow(image_initial_currents_A, cmap="RdBu_r", aspect="auto",
                      vmin=-image_initial_color_limit_A, vmax=image_initial_color_limit_A,
                      extent=(0.5, len(starts) + 0.5, 32.5, 0.5))
axes[0].set(xlabel="Imaging current state", ylabel="Calibration channel",
            title=f"Initial DAC commands: {IMAGE_INITIALIZATION}")
fig.colorbar(plot, ax=axes[0], label="Current [A]")
image_initial_settled_field_Hz = coil_maps @ image_initial_currents_A[:, 0]
plot = image_panel(axes[1], np.where(support, image_initial_settled_field_Hz, np.nan),
                   "Settled ACDC field for the initial commands [Hz]", cmap="RdBu_r")
fig.colorbar(plot, ax=axes[1])
show_figure(fig)
print("The MATLAB solver will receive these starting commands; the zero-current baseline is unchanged.")


## 10. Run the existing MATLAB forward model

The notebook writes a MAT request and calls `run_acdc_notebook_stage`. MATLAB evaluates
`forward_model_experiment` twice on exactly the selected object: once with B0 = 0 and once
with measured B0. Comparing those twins tests the simulated effect of the field.
The measured scan is displayed separately because reference/scan contrast and receive
sensitivity are not calibrated by this model.
The unit-step preview is also calculated in MATLAB. It checks the time constant and
the integrated current used for phase evolution before starting optimization.
Zero-current baseline twins are identical whether the response model is enabled or disabled.


In [ ]:
# The bridge only orchestrates the existing MATLAB functions.
request = {
    "input_file": str(INPUT_MAT), "calibration_file": str(CALIBRATION_MAT),
    "output_dir": str(WORK_DIR / "matlab"), "max_current_A": MAX_CURRENT_A,
    "total_current_limit_A": TOTAL_CURRENT_LIMIT_A, "field_limit_Hz": FIELD_LIMIT_HZ,
    "initial_currents_A": image_initial_currents_A,
    "max_iterations": float(MAX_ITERATIONS), "workers": float(WORKERS),
    "threads_per_worker": float(THREADS_PER_WORKER), "device": DEVICE, "scratch_dir": str(SCRATCH_DIR),
}
REQUEST_MAT = WORK_DIR / "matlab_request.mat"
savemat(REQUEST_MAT, {"request": request}, long_field_names=True)


def run_matlab_stage(stage):
    assert MATLAB_EXECUTABLE.is_file(), f"MATLAB executable not found: {MATLAB_EXECUTABLE}"
    quote = lambda path: str(path).replace("'", "''")
    command = f"addpath('{quote(EXPERIMENT_DIR)}'); run_acdc_notebook_stage('{quote(REQUEST_MAT)}','{stage}');"
    log_path = WORK_DIR / f"matlab_{stage}.log"
    print(f"Running MATLAB {stage}; full log: {log_path}", flush=True)
    environment = os.environ.copy()
    environment["TMPDIR"] = str(SCRATCH_DIR)
    with log_path.open("w") as log:
        completed = subprocess.run([str(MATLAB_EXECUTABLE), "-batch", command], stdout=log, stderr=subprocess.STDOUT, env=environment)
    print("\n".join(log_path.read_text().splitlines()[-12:]))
    assert completed.returncode == 0, f"MATLAB failed; inspect {log_path}."
    result_name = "forward.mat" if stage == "forward" else "optimization_stage.mat"
    result = loadmat(WORK_DIR / "matlab" / result_name, simplify_cells=True)
    assert result["status"]["state"] == "complete", result["status"]["message"]
    return result


forward = run_matlab_stage("forward") if RUN_FORWARD_MODEL else None
if forward is None:
    print("Forward stage disabled explicitly in the parameter cell.")
else:
    fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
    twin_scale = max(np.abs(forward["ideal_image"]).max(), np.abs(forward["measured_image"]).max())
    image_panel(axes[0], np.abs(forward["ideal_image"]), "MATLAB: ideal zero-B0 magnitude", vmin=0, vmax=twin_scale)
    image_panel(axes[1], np.abs(forward["measured_image"]), "MATLAB: measured-B0 prediction", vmin=0, vmax=twin_scale)
    image_panel(axes[2], np.abs(np.abs(forward["measured_image"]) - np.abs(forward["ideal_image"])), "Absolute magnitude difference", cmap="magma")
    image_panel(axes[3], scan_image, "Measured RSS scan: own scale")
    show_figure(fig)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
    for ax, values, title in zip(axes,
                                 (forward["ideal_kdata"], forward["measured_kdata"], forward["measured_kdata"] - forward["ideal_kdata"]),
                                 ("Ideal k-space", "Measured-B0 k-space", "Complex k-space difference")):
        image_panel(ax, np.log1p(np.abs(values)), title + ": log(1 + magnitude)", cmap="magma")
    show_figure(fig)
    print("Zero-current baseline magnitude relative MSE:", forward["relative_mse"])
    preview = forward["response_preview"]
    preview_time_us = np.asarray(preview["time_s"]).ravel() * 1e6
    step_currents = np.asarray(preview["actual_A"])
    step_integrals = np.asarray(preview["integrated_As"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
    axes[0].plot(preview_time_us, step_currents, alpha=0.5)
    axes[0].axhline(0.99, color="black", linestyle="--", label="99% of command")
    axes[0].set(xlabel="Time after command [us]", ylabel="Predicted current [A]",
                title="MATLAB response to a 0-to-1 A step")
    axes[0].legend()
    axes[1].plot(preview_time_us, step_integrals * 1e6, alpha=0.5)
    axes[1].plot(preview_time_us, preview_time_us, "k--", label="Instantaneous response")
    axes[1].set(xlabel="Time after command [us]", ylabel="Integrated current [A us]",
                title="Current integral used to calculate phase")
    axes[1].legend()
    show_figure(fig)
    show_table([
        ("First-order response enabled", bool(forward["current_response"]["enabled"])),
        ("Tau range [us]", [float(tau_us.min()), float(tau_us.max())]),
        ("First-order t99 range [us]", [float(tau_us.min() * np.log(100)), float(tau_us.max() * np.log(100))]),
        ("Initial actual current [A]", preview["initial_current_A"]),
    ])


## 11. MATLAB optimization and convergence

The optimizer minimizes magnitude image error to the zero-B0 twin. With the response
enabled, **every objective evaluation and its gradient use the integrated real current**;
the optimized variables remain the 32 DAC commands per imaging line.
Currents and the generated ACDC field are constrained at these commanded states.
The diagnostics below also check the predicted real currents and fields at ADC samples.
These sample checks do not certify limits between samples for different channel taus.
The numerical optimization can be inspected independently of wiring/trigger parameters.


In [ ]:
THREADS_PER_WORKER = 6
request["threads_per_worker"] = float(THREADS_PER_WORKER)
savemat(REQUEST_MAT, {"request": request}, long_field_names=True)

In [ ]:
# Run MATLAB optimization with the explicit numerical limits above.
optimization_stage = run_matlab_stage("optimize") if RUN_OPTIMIZATION else None
solution = None
if optimization_stage is not None:
    files = list((WORK_DIR / "matlab/optimization").glob("solution_*Hz.mat"))
    assert len(files) == 1, "Expected one solver result for the selected field limit."
    solution = loadmat(files[0], simplify_cells=True)
    saved_initial_currents_A = np.asarray(solution.get("initial_currents_A", []), dtype=float)
    saved_starts = np.asarray(solution["line_start_s"]).ravel()
    saved_ends = np.asarray(solution.get("line_end_s", [])).ravel()
    saved_response = solution.get("current_response", {})
    saved_tau_s = np.asarray(saved_response.get("tau_s", [])).ravel()
    response_matches = (
        bool(saved_response.get("enabled", False)) == CURRENT_RESPONSE_ENABLED
        and saved_tau_s.shape == (32,)
        and np.allclose(saved_tau_s, CURRENT_RESPONSE["tau_s"], atol=1e-15, rtol=0)
    )
    solution_checks = [
        ("Saved initial commands match the selected starting point", saved_initial_currents_A.shape == image_initial_currents_A.shape and np.allclose(saved_initial_currents_A, image_initial_currents_A, atol=1e-12, rtol=0)),
        ("Temporal response matches this run", response_matches),
        ("Registered calibration used", solution.get("calibration_mode") == "registered" and bool(solution.get("registration_confirmed"))),
        ("Current dimensions match this sequence", np.shape(solution["currents_A"]) == (32, len(starts))),
        ("Start times match this sequence", saved_starts.shape == starts.shape and np.allclose(saved_starts, starts, atol=1e-12, rtol=0)),
        ("Command end times match this sequence", not (CURRENT_RESPONSE_ENABLED or timing["sequence_kind"] == "gre") or (saved_ends.shape == ends.shape and np.allclose(saved_ends, ends, atol=1e-12, rtol=0))),
        ("Saved field bound matches this run", float(solution["field_limit_Hz"]) == FIELD_LIMIT_HZ),
    ]
    show_table([(name, "PASS" if passed else "FAIL") for name, passed in solution_checks], ("Saved result check", "Result"))
    assert all(passed for _, passed in solution_checks), "Saved MATLAB results do not match these inputs; rerun from the parameter cell."
    info = solution["optimization_info"]
    show_table([(key, info[key]) for key in ("exitflag", "fval", "converged", "current_limits_met", "field_limits_met")])
    show_table([
        ("Image optimizer initialization", IMAGE_INITIALIZATION),
        ("Magnitude relative MSE at the requested initial commands", float(solution["initial_relative_mse"])),
        ("Final magnitude relative MSE", float(info["fval"])),
    ])
    history = solution["history"]
    iterations = np.atleast_1d(history["iteration"])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")
    for ax, key, title in zip(axes, ("fval", "firstorderopt", "constrviolation"),
                              ("Magnitude relative MSE", "First-order optimality", "Constraint violation")):
        values = np.atleast_1d(history[key])
        ax.plot(iterations, values, ".-")
        ax.set(xlabel="Iteration", ylabel=title, title=title)
    show_figure(fig)
else:
    print("Optimization disabled explicitly in the parameter cell.")

## 12. Corrected image, DAC commands and nominal fields

The corrected image and k-space below are the MATLAB outputs with the selected response
model. Their error is relative to the simulated ideal object; it is not a measured hardware
improvement. The state plots show DAC commands and their **nominal steady-state fields**.
The field bound is `FIELD_LIMIT_HZ`, applied to the generated ACDC field, rather than to
the residual B0 + ACDC field. Step 12b displays predicted real currents and fields.


In [ ]:
# Inspect solver outputs even when convergence is not achieved.
if solution is not None:
    ideal = solution["reference"]
    corrected = solution["image"]
    baseline = loadmat(WORK_DIR / "matlab/optimization/baseline.mat", simplify_cells=True)
    before = baseline["initial_image"]
    scale = max(np.abs(ideal).max(), np.abs(before).max(), np.abs(corrected).max())
    fig, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
    for ax, values, title in zip(axes[:3], (ideal, before, corrected),
                                 ("Ideal zero-B0 twin", "Initial measured-B0 twin", "Optimized ACDC twin")):
        image_panel(ax, np.abs(values), title, vmin=0, vmax=scale)
    image_panel(axes[3], np.abs(np.abs(corrected) - np.abs(ideal)), "Final magnitude difference", cmap="magma")
    show_figure(fig)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
    for ax, values, title in zip(axes,
                                 (baseline["initial_kdata"], solution["kdata"], solution["kdata"] - baseline["initial_kdata"]),
                                 ("Initial k-space", "Optimized k-space", "Correction-induced k-space change")):
        image_panel(ax, np.log1p(np.abs(values)), title + ": log(1 + magnitude)", cmap="magma")
    show_figure(fig)
    currents = solution["currents_A"]
    generated_fields = solution["generated_field_Hz"]
    maximum_field = np.max(np.abs(generated_fields[support]), axis=0)
    total_current = np.abs(currents).sum(axis=0)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout="constrained")
    plot = image_panel(axes[0], currents, "DAC command per channel and imaging state [A]", cmap="RdBu_r")
    axes[0].set(xlabel="Chronological current state", ylabel="Calibration channel")
    fig.colorbar(plot, ax=axes[0])
    axes[1].plot(total_current)
    axes[1].axhline(TOTAL_CURRENT_LIMIT_A, color="red", linestyle="--", label="Total bound")
    axes[1].set(xlabel="Current state", ylabel="Sum |I| [A]", title="Total current per state")
    axes[1].legend()
    axes[2].plot(maximum_field)
    axes[2].axhline(FIELD_LIMIT_HZ, color="red", linestyle="--", label="Generated-field bound")
    axes[2].set(xlabel="Current state", ylabel="Max |ACDC field| [Hz]", title="Nominal commanded field on ROI")
    axes[2].legend()
    show_figure(fig)
    chosen_states = np.unique(np.linspace(0, currents.shape[1] - 1, 4).astype(int))
    residual_fields = b0_on_reference[..., None] + generated_fields
    residual_scale = np.nanmax(np.abs(np.r_[b0_on_reference[support], residual_fields[support].ravel()]))
    fig, axes = plt.subplots(2, len(chosen_states), figsize=(4 * len(chosen_states), 8), layout="constrained", squeeze=False)
    for column, state in enumerate(chosen_states):
        plot = image_panel(axes[0, column], np.where(support, generated_fields[..., state], np.nan),
                           f"Nominal ACDC field, state {state + 1} [Hz]", cmap="RdBu_r", vmin=-FIELD_LIMIT_HZ, vmax=FIELD_LIMIT_HZ)
        fig.colorbar(plot, ax=axes[0, column])
        plot = image_panel(axes[1, column], np.where(support, residual_fields[..., state], np.nan),
                           f"B0 + ACDC, state {state + 1} [Hz]", cmap="RdBu_r", vmin=-residual_scale, vmax=residual_scale)
        fig.colorbar(plot, ax=axes[1, column])
    show_figure(fig)
    current_ok = np.all(np.abs(currents) <= MAX_CURRENT_A + 1e-8) and np.all(total_current < TOTAL_CURRENT_LIMIT_A)
    field_ok = np.max(maximum_field) <= FIELD_LIMIT_HZ + 1e-8
    quantized_field_max = np.max(np.abs(coil_maps[support] @ quantize_currents(currents)))
    quantized_field_ok = quantized_field_max <= FIELD_LIMIT_HZ + 1e-8
    #solver_ok = bool(info["converged"]) and float(info["exitflag"]) > 0 and current_ok and field_ok and quantized_field_ok
    solver_ok = current_ok and field_ok and quantized_field_ok
    show_table([("Recomputed current constraints", current_ok), ("Recomputed field constraint", field_ok),
                ("Nominal quantized field maximum [Hz]", quantized_field_max), ("Solver eligible for export", solver_ok)])
else:
    solver_ok = False

## 12b. Real channel response and phase history

MATLAB predicts the actual current at **each ADC sample**, with no reset at a new line.
The zoom shows continuity through a command change. The phase plot includes all earlier
transitions, so a current that has settled by the ADC can still change the image.
The image comparison uses **the same DAC commands** with and without the response;
it isolates the simulated temporal effect and is not a second optimization.
The field snapshots use the ADC sample nearest the center of each chronological line.


In [ ]:
# Plot MATLAB response diagnostics; all current dynamics and phases are calculated in MATLAB.
if solution is not None:
    response = solution["response_diagnostics"]
    adc_time_s = np.asarray(response["adc_time_s"]).ravel()
    chronological = np.argsort(adc_time_s)
    commanded_adc = np.asarray(response["commanded_currents_at_adc_A"])
    actual_adc = np.asarray(response["actual_currents_at_adc_A"])
    dense_time_s = np.asarray(response["dense_time_s"]).ravel()
    dense_command = np.asarray(response["dense_commanded_currents_A"])
    dense_actual = np.asarray(response["dense_actual_currents_A"])
    channel = int(np.argmax(np.max(np.abs(dense_actual - dense_command), axis=1)))
    fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout="constrained")
    for ax, values, title in zip(axes[:2], (commanded_adc, actual_adc),
                                 ("DAC commands at ADC", "Predicted real currents at ADC")):
        plot = ax.imshow(values[:, chronological], aspect="auto", origin="lower", cmap="RdBu_r",
                         extent=[adc_time_s.min() * 1000, adc_time_s.max() * 1000, 0.5, 32.5],
                         vmin=-MAX_CURRENT_A, vmax=MAX_CURRENT_A)
        ax.set(xlabel="Time from RF center [ms]", ylabel="Calibration channel", title=title)
        fig.colorbar(plot, ax=ax, label="Current [A]")
    zoom_origin_s = dense_time_s[0]
    zoom_time_us = (dense_time_s - zoom_origin_s) * 1e6
    axes[2].step(zoom_time_us, dense_command[channel], where="post", label="DAC command")
    axes[2].plot(zoom_time_us, dense_actual[channel], label="Predicted real current")
    axes[2].set(xlabel=f"Time from zoom start ({zoom_origin_s * 1000:.3f} ms after RF) [us]",
                ylabel="Current [A]", title=f"Transition zoom: channel {channel + 1}")
    axes[2].legend()
    show_figure(fig)

    # Coil-map projection only: integrated currents themselves come from MATLAB.
    pixel = tuple(int(index) for index in np.unravel_index(
        np.argmax(np.where(support, np.abs(b0_on_reference), -np.inf)), support.shape))
    voxel_coils = coil_maps[pixel]
    real_phase_rad = -2 * np.pi * (voxel_coils @ response["integrated_currents_to_adc_As"])
    instantaneous_phase_rad = -2 * np.pi * (voxel_coils @ response["commanded_integrated_currents_to_adc_As"])
    fig, axes = plt.subplots(1, 3, figsize=(17, 4), layout="constrained")
    for values, label in ((instantaneous_phase_rad, "Instantaneous commands"), (real_phase_rad, "Selected response")):
        axes[0].plot(adc_time_s[chronological] * 1000, values[chronological], label=label)
    axes[0].set(xlabel="Time from RF center [ms]", ylabel="Accumulated ACDC phase [rad]",
                title=f"Phase history at ROI pixel {pixel}")
    axes[0].legend()
    axes[1].plot(adc_time_s[chronological] * 1000,
                 (real_phase_rad - instantaneous_phase_rad)[chronological])
    axes[1].set(xlabel="Time from RF center [ms]", ylabel="Phase difference [rad]",
                title="Temporal-response phase minus instantaneous phase")
    for values, label in ((commanded_adc, "Nominal command field"), (actual_adc, "Predicted real field")):
        total_field = b0_on_reference[pixel] + voxel_coils @ values
        axes[2].plot(adc_time_s[chronological] * 1000, total_field[chronological], label=label)
    axes[2].set(xlabel="Time from RF center [ms]", ylabel="B0 + ACDC [Hz]",
                title="Field at every ADC sample at the same ROI pixel")
    axes[2].legend()
    show_figure(fig)

    instantaneous_image = response["instantaneous_image"]
    comparison_scale = max(np.abs(instantaneous_image).max(), np.abs(solution["image"]).max())
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
    image_panel(axes[0], np.abs(instantaneous_image), "Same commands: instantaneous response", vmin=0, vmax=comparison_scale)
    image_panel(axes[1], np.abs(solution["image"]), "Same commands: selected response", vmin=0, vmax=comparison_scale)
    plot = image_panel(axes[2], np.abs(solution["image"]) - np.abs(instantaneous_image),
                       "Temporal response: magnitude difference", cmap="RdBu_r")
    fig.colorbar(plot, ax=axes[2])
    show_figure(fig)

    real_fields = response["generated_field_at_line_center_Hz"]
    real_residuals = b0_on_reference[..., None] + real_fields
    real_scale = np.nanmax(np.abs(real_residuals[support]))
    fig, axes = plt.subplots(2, len(chosen_states), figsize=(4 * len(chosen_states), 8), layout="constrained", squeeze=False)
    for column, state in enumerate(chosen_states):
        center_ms = np.asarray(response["line_center_time_s"]).ravel()[state] * 1000
        plot = image_panel(axes[0, column], np.where(support, real_fields[..., state], np.nan),
                           f"Real ACDC: line {state + 1}, {center_ms:.2f} ms [Hz]", cmap="RdBu_r",
                           vmin=-FIELD_LIMIT_HZ, vmax=FIELD_LIMIT_HZ)
        fig.colorbar(plot, ax=axes[0, column])
        plot = image_panel(axes[1, column], np.where(support, real_residuals[..., state], np.nan),
                           f"Real B0 + ACDC: line {state + 1} [Hz]", cmap="RdBu_r", vmin=-real_scale, vmax=real_scale)
        fig.colorbar(plot, ax=axes[1, column])
    show_figure(fig)

    actual_current_ok = (np.all(np.abs(actual_adc) <= MAX_CURRENT_A + 1e-8)
                         and np.all(np.abs(actual_adc).sum(axis=0) < TOTAL_CURRENT_LIMIT_A))
    actual_field_ok = bool(response["actual_adc_field_limits_met"])
    solver_ok = solver_ok and actual_current_ok and actual_field_ok
    show_table([
        ("Same-command instantaneous relative MSE", response["instantaneous_relative_mse"]),
        ("Selected-response relative MSE", info["fval"]),
        ("Maximum real generated field at ADC [Hz]", response["maximum_actual_quality_field_Hz"]),
        ("Predicted real current limits at ADC", actual_current_ok),
        ("Predicted real field limit at ADC", actual_field_ok),
        ("Solver eligible for CSV export", solver_ok),
    ])
else:
    print("Response diagnostics require a MATLAB optimization result.")


## 12c. Preview the predicted real waveform

This figure shows the **complete current waveform predicted by the selected temporal
response**, including every imaging-line transition and the decay after the final zero
command. MATLAB uses the optimized DAC setpoints, the sequence command windows and
the saved channel time constants. The initial real current is assumed to be zero.
This preview can be run on an existing solution without repeating optimization.

The first panel displays all physical outputs. The second panel zooms into a transition
so the first-order rise or fall is visible. `WAVEFORM_PREVIEW_CHANNEL` uses the 1-based
output numbers in `channel_map`; `None` selects the largest command change.
The channel order, polarity and time origin follow the settings used in section 13.
With the temporal model disabled, this preview follows instantaneous commands.

The waveform is a model prediction requiring experimental validation. Section 13
below keeps its existing DAC-command preview and CSV export.


In [ ]:
# Sample the full predicted waveform in MATLAB; do not optimize or export here.
WAVEFORM_PREVIEW_CHANNEL = None  # Configured output number (1-based), or None for the largest jump.

if solution is None:
    print("A MATLAB solution is needed for the predicted waveform preview.")
else:
    from export_acdc_waveform import validate_waveform_settings

    assert np.isscalar(RF_DELAY_S) and np.isfinite(RF_DELAY_S) and RF_DELAY_S >= 0, "RF_DELAY_S must be finite and nonnegative."
    preview_currents = np.asarray(solution["currents_A"])
    preview_settings = validate_waveform_settings(WAVEFORM_PARAMETERS, preview_currents.shape[0])
    preview_request = {
        "output_dir": str(WORK_DIR / "matlab"),
        "waveform_currents_A": preview_currents,
        "waveform_line_start_s": starts,
        "waveform_line_end_s": ends,
        "waveform_current_response": solution["current_response"],
    }
    preview_request_file = WORK_DIR / "waveform_preview_request.mat"
    savemat(preview_request_file, {"request": preview_request}, long_field_names=True)
    # Launch only the waveform sampler, so this cell also works in an already-running kernel.
    assert MATLAB_EXECUTABLE.is_file(), f"MATLAB executable not found: {MATLAB_EXECUTABLE}"
    quote = lambda path: str(path).replace("'", "''")
    matlab_command = f"addpath('{quote(EXPERIMENT_DIR)}'); run_acdc_notebook_stage('{quote(preview_request_file)}','waveform');"
    preview_log = WORK_DIR / "matlab_waveform_preview.log"
    print(f"Sampling the waveform in MATLAB; full log: {preview_log}", flush=True)
    environment = os.environ.copy()
    environment["TMPDIR"] = str(SCRATCH_DIR)
    with preview_log.open("w") as log:
        completed = subprocess.run([str(MATLAB_EXECUTABLE), "-batch", matlab_command],
                                   stdout=log, stderr=subprocess.STDOUT, env=environment)
    assert completed.returncode == 0, f"MATLAB failed; inspect {preview_log}."
    waveform_preview = loadmat(WORK_DIR / "matlab/waveform_preview.mat", simplify_cells=True)
    assert waveform_preview["status"]["state"] == "complete", waveform_preview["status"]["message"]
    model_time_s = np.asarray(waveform_preview["waveform_time_s"]).ravel()
    predicted_current = np.asarray(waveform_preview["waveform_actual_currents_A"])
    commanded_current = np.asarray(waveform_preview["waveform_commanded_currents_A"])

    # Apply the same output numbering and polarity as the CSV; unused outputs stay zero.
    output_numbers = np.asarray(preview_settings["channel_map"])
    polarity = np.asarray(preview_settings["channel_polarity"])[:, None]
    output_count = ((int(output_numbers.max()) + 7) // 8) * 8
    physical_real = np.zeros((output_count, model_time_s.size))
    physical_command = np.zeros_like(physical_real)
    physical_real[output_numbers - 1] = predicted_current * polarity
    physical_command[output_numbers - 1] = commanded_current * polarity
    plot_time_s = model_time_s + RF_DELAY_S
    if RF_DELAY_S > 0:
        plot_time_s = np.r_[0.0, plot_time_s]
        physical_real = np.column_stack((np.zeros(output_count), physical_real))
        physical_command = np.column_stack((np.zeros(output_count), physical_command))

    # Include the first command and final return to zero when choosing the zoom.
    jumps = np.abs(np.diff(np.column_stack((np.zeros(preview_currents.shape[0]), commanded_current)), axis=1))
    if WAVEFORM_PREVIEW_CHANNEL is None:
        channel, transition_sample = np.unravel_index(np.argmax(jumps), jumps.shape)
    else:
        assert WAVEFORM_PREVIEW_CHANNEL in output_numbers, "Choose an output number present in channel_map."
        channel = int(np.flatnonzero(output_numbers == WAVEFORM_PREVIEW_CHANNEL)[0])
        transition_sample = int(np.argmax(jumps[channel]))
    output_index = int(output_numbers[channel] - 1)
    transition_s = model_time_s[transition_sample] + RF_DELAY_S
    saved_tau_s = np.asarray(solution["current_response"]["tau_s"]).ravel()
    channel_tau_s = float(saved_tau_s[0] if saved_tau_s.size == 1 else saved_tau_s[channel])
    zoom = np.abs(plot_time_s - transition_s) <= 5 * channel_tau_s + 1e-12
    origin = "first RF center" if RF_DELAY_S == 0 else "external trigger"
    response_enabled = bool(solution["current_response"]["enabled"])

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), layout="constrained")
    axes[0].plot(plot_time_s * 1000, physical_real.T, alpha=0.65, linewidth=0.8)
    axes[0].axvline((ends[-1] + RF_DELAY_S) * 1000, color="black", linestyle="--", label="Final zero command")
    axes[0].set(xlabel=f"Time from {origin} [ms]", ylabel="Physical output current [A]",
                title="Complete predicted current waveform" if response_enabled else "Complete waveform: temporal response disabled")
    axes[0].legend()
    zoom_time_us = (plot_time_s[zoom] - transition_s) * 1e6
    axes[1].step(zoom_time_us, physical_command[output_index, zoom], where="post",
                 color="gray", linestyle="--", label="DAC command")
    axes[1].plot(zoom_time_us, physical_real[output_index, zoom], label="Predicted real current")
    axes[1].set(xlabel="Time from selected command change [us]", ylabel="Current [A]",
                title=f"Output {output_index + 1}: transition at {transition_s * 1000:.3f} ms; tau = {channel_tau_s * 1e6:.2f} us")
    axes[1].legend()
    show_figure(fig)


## 13. Preview and export one waveform CSV

The CSV contains time in seconds (or zeros with `CSV_ZERO_TIME_COLUMN = True`),
followed by currents in amperes in physical output order. The preview keeps the real times.
It starts at time zero with zero currents and ends with a return to zero. For GRE, zero
states are inserted between acquired lines so the current is zero during RF excitations.
EPI retains its chronological imaging states.

With `RF_DELAY_S = 0`, timestamps are relative to the first RF center. Enter a measured
trigger-to-RF delay to express them relative to that trigger instead. No trigger is sent.
`channel_map` and `channel_polarity` define the CSV columns; no hardware is configured.

Each export creates a new numbered CSV in the same folder: `waveform1.csv`, `waveform2.csv`, etc.
The next number follows the largest existing numeric suffix; previous CSVs remain intact.
No scripts, manifests or controller commands are generated. The figure below previews the currents when export succeeds.
The CSV always contains the optimized **DAC commands**. The predicted real currents
are diagnostic outputs kept in this notebook; exporting them would apply the response twice.
The temporal model uses the existing sequence-derived command windows and adds no
trigger delay. Verify the time origin independently for the acquisition.


In [ ]:
# Option de la première colonne du CSV.
CSV_ZERO_TIME_COLUMN = True  # True : des 0 ; False : temps en secondes

# Recharger l'export pour utiliser la numérotation dans un noyau déjà ouvert.
import importlib
import export_acdc_waveform
export_waveform = importlib.reload(export_acdc_waveform).export_waveform

# Generate data only. The FPGA is operated separately from this notebook.
csv_file = None
if EXPORT_WAVEFORM and solver_ok:
    assert np.isscalar(RF_DELAY_S) and np.isfinite(RF_DELAY_S) and RF_DELAY_S >= 0, "RF_DELAY_S must be finite and nonnegative."
    if timing["sequence_kind"] == "gre":
        csv_edges = np.column_stack((starts, ends)).ravel() + RF_DELAY_S
        csv_currents = np.zeros((currents.shape[0], 2 * currents.shape[1] - 1))
        csv_currents[:, ::2] = currents
        for start, end in zip(starts, ends):
            overlaps_rf = (np.asarray(timing["rf_start_s"]) < end - 1e-12) & (np.asarray(timing["rf_end_s"]) > start + 1e-12)
            assert not overlaps_rf.any(), "A current window overlaps a GRE RF pulse."
    else:
        csv_edges = np.r_[starts, ends[-1]] + RF_DELAY_S
        csv_currents = currents
    try:
        csv_export = export_waveform(
            csv_currents, csv_edges, WAVEFORM_PARAMETERS, OUTPUT_DIR,
            zero_time_column=CSV_ZERO_TIME_COLUMN,
            numbered_filename=True,
        )
        csv_file = Path(csv_export["csv_file"])
    except (ValueError, TypeError, FileExistsError) as error:
        print("CSV export blocked:", error)
else:
    print("CSV export disabled or numerical solution not eligible.")

if csv_file is not None:
    waveform = np.loadtxt(csv_file, delimiter=",")
    # Le graphique conserve les vrais temps dans les deux modes d'export.
    preview_times = np.r_[0.0, csv_edges] if csv_edges[0] > 0 else csv_edges
    fig, ax = plt.subplots(figsize=(12, 4), layout="constrained")
    ax.step(preview_times * 1000, waveform[:, 1:], where="post", alpha=0.6)
    origin = "first RF center" if RF_DELAY_S == 0 else "external trigger"
    ax.set(xlabel=f"Time from {origin} [ms]", ylabel="Physical output current [A]", title="Exported waveform: initial and final zero currents")
    show_figure(fig)
    print("New exported file:", csv_file)
    print("Première colonne :", "uniquement des 0" if CSV_ZERO_TIME_COLUMN else "temps en secondes")
print("Figures remain in this notebook. MATLAB working files are temporary.")


## Reading the result

1. Inspect the echo magnitudes, phase consistency and B0 components before trusting the field.
2. Compare the requested ROI with B0 coverage. The ROI is never reduced automatically.
3. Check the MATLAB images, solver history and current/field constraints.
4. Enter the CSV column mapping and desired time origin, then inspect the waveform preview.

Save the notebook to preserve its diagnostic figures. Each exported result is saved as a new numbered waveform CSV.
The MATLAB inputs, logs and results live in a temporary workspace for the current kernel.
The notebook does not upload a waveform, send a trigger or operate the FPGA.

## 14. Compare the measured scanner result

After applying the waveform on the scanner, place the corrected acquisition in
`INPUT_DIR / "results" / "scan_corr.dat"`. This section compares **three measured scans**:
the GRE reference, the scan without ACDC and the scan acquired with ACDC.
It does not rerun MATLAB or modify the exported waveform.

The reference and baseline reconstructions are reused from steps 4 and 5. The corrected
Twix is reconstructed independently, with its own navigators when present. ACDC shimming
on the scanner and offline navigator phase correction are different operations.
If the corrected file is absent, this section reports it and skips the comparison.

The parameters below control only this comparison. With `RMSE_USE_ROI = True`, the RMSE
uses the **final ROI used for optimization** (`support`, including any validity filtering
chosen earlier). Set it to `False` to use the full common image coverage. The exact mask
and excluded pixels are shown before computing the error.


In [ ]:
# Measured results are needed only after the scanner experiment.
RESULTS_DIR = INPUT_DIR / "results"
CORRECTED_SCAN_FILE = RESULTS_DIR / "scan_corr.dat"
CORRECTED_SEQUENCE_FILE = FILES["scan_sequence"]  # Change if the corrected scan used another sequence.
RMSE_USE_ROI = True
NORMALIZE_COMPARISON = True  # One baseline-to-reference gain shared by both scans.

print("Measured corrected scan:", CORRECTED_SCAN_FILE)
print("Corrected scan sequence:", CORRECTED_SEQUENCE_FILE)


### 14a. Reconstruct the corrected acquisition and verify its sequence

`read_scan` handles the Twix `REFLECT` and `PHASCOR` flags and combines all receivers
with RSS. Its navigator fit comes from `scan_corr.dat`; the baseline fit is not reused.
The same `NAVIGATOR_PHASE_BRANCH` is applied to both EPI scans.
The corrected acquisition must match its own `.seq`. The three scans must describe the
same slice center and orientation before a pixelwise RMSE can be computed.

The before/after panels below diagnose navigator processing within the corrected
acquisition. They are not the comparison of ACDC off versus ACDC on.


In [ ]:
measured_corrected_image = None
if not CORRECTED_SCAN_FILE.is_file():
    print(f"Comparison skipped: acquire the corrected scan and place it at {CORRECTED_SCAN_FILE}.")
    print("Then rerun section 14 from its parameter cell.")
else:
    assert CORRECTED_SCAN_FILE.suffix.lower() == ".dat", "The corrected scan must be raw Siemens Twix data (.dat)."
    assert CORRECTED_SEQUENCE_FILE.is_file(), f"Missing corrected sequence: {CORRECTED_SEQUENCE_FILE}"
    measured_corrected_acquisition, measured_corrected_timing = read_sequence(CORRECTED_SEQUENCE_FILE)
    measured_corrected_candidate, measured_corrected_metadata, measured_corrected_diagnostics = read_scan(
        CORRECTED_SCAN_FILE, measured_corrected_acquisition, return_intermediates=True,
        navigator_phase_branch=NAVIGATOR_PHASE_BRANCH)
    assert np.isfinite(measured_corrected_candidate).all() and measured_corrected_candidate.max() > 0, "Corrected reconstruction has invalid or zero signal."

    corrected_checks = []
    for key in ("TE_s", "TR_s", "dwell_s"):
        tolerance = 1e-9 if key == "dwell_s" else 1e-6
        corrected_checks.append((key, abs(measured_corrected_metadata[key] - measured_corrected_acquisition[key]) <= tolerance))
    corrected_checks += [
        ("FOV", np.allclose(measured_corrected_metadata["FOV_m"], measured_corrected_acquisition["FOV_m"], rtol=1e-4, atol=1e-6)),
        ("Navigator count", measured_corrected_metadata["navigator_count"] == measured_corrected_timing["navigator_count"]),
        ("Readout polarity", np.array_equal(measured_corrected_metadata["polarity"], measured_corrected_acquisition["polarity"])),
        ("Chronological phase rows", measured_corrected_metadata["acquired_phase_rows"] == np.argsort(measured_corrected_acquisition["line_index"][:, 0]).tolist()),
    ]
    show_table([(name, "PASS" if passed else "FAIL") for name, passed in corrected_checks], ("Corrected acquisition check", "Result"))
    assert all(passed for _, passed in corrected_checks), "Corrected Twix and sequence disagree; inspect the checks above."

    for label, metadata in (("Baseline", scan_metadata), ("Corrected", measured_corrected_metadata)):
        assert np.allclose(metadata["slice_position_mm"], reference_metadata["slice_position_mm"], rtol=0, atol=1e-3), f"{label} and reference slice centers differ; register the acquisitions before RMSE."
        assert np.isclose(abs(np.dot(metadata["slice_quaternion_wxyz"], reference_metadata["slice_quaternion_wxyz"])), 1, rtol=0, atol=1e-8), f"{label} and reference orientations differ; register the acquisitions before RMSE."

    measured_corrected_image = measured_corrected_candidate
    measured_metadata = (reference_metadata, scan_metadata, measured_corrected_metadata)
    show_table([
        ("Matrix [phase, readout]", *[metadata["image_shape"] for metadata in measured_metadata]),
        ("TE [ms]", *[metadata["TE_s"] * 1000 for metadata in measured_metadata]),
        ("TR [ms]", *[metadata["TR_s"] * 1000 for metadata in measured_metadata]),
        ("Slice thickness [mm]", *[metadata["slice_thickness_mm"] for metadata in measured_metadata]),
        ("Navigators", *[metadata["navigator_count"] for metadata in measured_metadata]),
    ], ("Measured parameter", "GRE reference", "Without ACDC", "With ACDC"))

    if measured_corrected_metadata["navigator_count"]:
        before_nav = measured_corrected_diagnostics["uncorrected_image"]
        nav_scale = max(before_nav.max(), measured_corrected_image.max())
        fig, axes = plt.subplots(1, 3, figsize=(16, 4), layout="constrained")
        image_panel(axes[0], before_nav, "ACDC on: before navigator correction", vmin=0, vmax=nav_scale)
        image_panel(axes[1], measured_corrected_image, "ACDC on: after navigator correction", vmin=0, vmax=nav_scale)
        plot = image_panel(axes[2], measured_corrected_diagnostics["navigator_phase_correction_rad"], "ACDC on: fitted navigator phase [rad]", cmap="RdBu_r")
        axes[2].set(xlabel="Readout column", ylabel="Receiver index")
        fig.colorbar(plot, ax=axes[2])
        show_figure(fig)
    else:
        print("Corrected scan has no navigators; no navigator phase correction is applied.")


### 14b. Put both scans on the simulation grid and show the RMSE mask

The native image grids come from their Twix matrix, FOV and slice center. The existing
`resample_to_grid` function samples each scan on the GRE physical grid when needed.
This is linear interpolation of already aligned images, with **no registration or extrapolation**.
Identical grids retain their original values.

Both errors use exactly the same pixels: the chosen ROI intersected with valid coverage
from both scans. The table reports any excluded pixels; this comparison mask does not
change the ROI used by MATLAB or the exported waveform.


In [ ]:
def measured_scan_on_reference(image, metadata):
    # Axial readout = patient +y; phase = patient -x, as verified in step 14a.
    ny, nx = image.shape
    dx, dy = np.asarray(metadata["FOV_m"]) / [nx, ny]
    sag, cor, _ = np.asarray(metadata["slice_position_mm"]) / 1000
    x = (np.arange(nx) - nx // 2) * dx + cor
    y = (np.arange(ny) - ny // 2) * dy - sag
    scan_x, scan_y = np.meshgrid(x, y)
    return resample_to_grid(image, scan_x, scan_y, np.isfinite(image), X_m, Y_m)


if measured_corrected_image is not None:
    measured_baseline_on_reference, measured_baseline_valid = measured_scan_on_reference(scan_image, scan_metadata)
    measured_corrected_on_reference, measured_corrected_valid = measured_scan_on_reference(measured_corrected_image, measured_corrected_metadata)
    requested_rmse_mask = support.copy() if RMSE_USE_ROI else np.ones(reference_magnitude.shape, dtype=bool)
    common_image_coverage = measured_baseline_valid & measured_corrected_valid & np.isfinite(reference_magnitude)
    measured_rmse_mask = requested_rmse_mask & common_image_coverage
    assert measured_rmse_mask.any(), "No common valid pixels remain for RMSE; inspect the image coverage and ROI."
    show_table([
        ("Requested region", "Final optimization ROI" if RMSE_USE_ROI else "Full reference image"),
        ("Requested pixels", int(requested_rmse_mask.sum())),
        ("Pixels excluded for missing scan coverage", int((requested_rmse_mask & ~common_image_coverage).sum())),
        ("Common pixels used for both RMSE values", int(measured_rmse_mask.sum())),
    ])
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), layout="constrained")
    image_panel(axes[0], requested_rmse_mask, "Requested RMSE region", vmin=0, vmax=1)
    image_panel(axes[1], common_image_coverage, "Common valid image coverage", vmin=0, vmax=1)
    image_panel(axes[2], reference_magnitude, "GRE: green = pixels used for both errors")
    axes[2].imshow(np.ma.masked_where(~measured_rmse_mask, measured_rmse_mask), cmap="Greens", vmin=0, vmax=1, alpha=0.45, interpolation="nearest")
    show_figure(fig)


### 14c. Measure RMSE and compare the three measured images

With `NORMALIZE_COMPARISON = True`, a single positive least-squares intensity gain is
estimated **only from the scan without ACDC** on the common RMSE mask:
`gain = sum(reference * baseline) / sum(baseline**2)`.
The same gain is applied unchanged to the scan with ACDC. All three images are then
divided by the GRE maximum within that mask. The RMSE and the final figure use these
same images, with one shared intensity scale. The corrected scan is never fitted or
normalized independently.

`RMSE = sqrt(mean((scan - reference)**2))` on the displayed mask.
`NRMSE [%] = 100 * RMSE / sqrt(mean(reference**2))` on that same mask.
Set `NORMALIZE_COMPARISON = False` to keep raw RSS intensities: RMSE is then in arbitrary
Twix intensity units. NRMSE is dimensionless in either mode. Native reconstructions remain unchanged.

These are **measured image differences**, separate from MATLAB's simulated optimization
MSE. GRE and EPI can differ in TE, TR, contrast, receive gain and slice thickness; those
differences also contribute to RMSE. A lower value measures closer image similarity
under this comparison, and does not isolate the shimming error.


In [ ]:
if measured_corrected_image is not None:
    reference_roi = reference_magnitude[measured_rmse_mask]
    baseline_roi = measured_baseline_on_reference[measured_rmse_mask]
    assert np.max(reference_roi) > 0 and np.sum(baseline_roi**2) > 0, "Reference and baseline need nonzero signal in the RMSE mask."
    measured_intensity_gain = float(np.dot(reference_roi, baseline_roi) / np.dot(baseline_roi, baseline_roi)) if NORMALIZE_COMPARISON else 1.0
    measured_reference_scale = float(reference_roi.max()) if NORMALIZE_COMPARISON else 1.0
    assert np.isfinite(measured_intensity_gain) and measured_intensity_gain > 0, "The baseline-to-reference intensity gain must be positive and finite."

    measured_comparison_images = [
        reference_magnitude / measured_reference_scale,
        measured_intensity_gain * measured_baseline_on_reference / measured_reference_scale,
        measured_intensity_gain * measured_corrected_on_reference / measured_reference_scale,
    ]
    reference_for_rmse = measured_comparison_images[0][measured_rmse_mask]
    reference_rms = np.sqrt(np.mean(reference_for_rmse**2))
    measured_rmse = [float(np.sqrt(np.mean((image[measured_rmse_mask] - reference_for_rmse)**2))) for image in measured_comparison_images[1:]]
    measured_nrmse_percent = [100 * error / reference_rms for error in measured_rmse]
    rmse_units = "GRE-normalized units" if NORMALIZE_COMPARISON else "Twix RSS units"
    show_table([
        ("Shared baseline-to-reference gain", f"{measured_intensity_gain:.6g}"),
        ("Reference normalization scale", f"{measured_reference_scale:.6g}"),
        ("RMSE units", rmse_units),
    ])
    show_table([
        (label, f"{error:.6f}", f"{relative_error:.3f}")
        for label, error, relative_error in zip(("Without ACDC", "With ACDC"), measured_rmse, measured_nrmse_percent)
    ], ("Measured scan vs GRE", "RMSE", "NRMSE [%]"))

    # Show the same physical grid and intensity scaling used in the error calculation.
    dx_mm = (X_m[0, 1] - X_m[0, 0]) * 1000
    dy_mm = (Y_m[1, 0] - Y_m[0, 0]) * 1000
    comparison_extent = [X_m.min() * 1000 - dx_mm / 2, X_m.max() * 1000 + dx_mm / 2,
                         Y_m.min() * 1000 - dy_mm / 2, Y_m.max() * 1000 + dy_mm / 2]
    comparison_max = max(np.nanmax(image) for image in measured_comparison_images)
    titles = ["GRE reference", f"Scan without ACDC\nRMSE = {measured_rmse[0]:.4f}", f"Scan with ACDC\nRMSE = {measured_rmse[1]:.4f}"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), layout="constrained", sharex=True, sharey=True)
    for ax, image, title in zip(axes, measured_comparison_images, titles):
        plot = ax.imshow(image, cmap="gray", vmin=0, vmax=comparison_max, origin="lower",
                         extent=comparison_extent, interpolation="nearest", aspect="equal")
        ax.set(title=title, xlabel="X: readout [mm]", ylabel="Y: phase [mm]")
    fig.colorbar(plot, ax=axes, label=f"Magnitude [{rmse_units}]")
    show_figure(fig)
